# Paper 4 — author careers and the supervision network

Figures from the hand-collected career metadata in `base_institucional.xlsx`, read through
the cleaning pipeline in `paper4/institutional/`. Nothing here touches the spreadsheet;
it reads the four tables that pipeline writes to `institutional/out/`.

**Run `institutional/build_clean_base.py` first** if the spreadsheet has changed.

Every figure follows economics-journal conventions (AER / QJE / Econometrica): grayscale
marks with black edges, serif type sized for the printed page, no title or source text
inside the plot. Each figure is saved to its own file with a `<stem>.caption.md` beside it.
Rules and rationale: `.claude/skills/styling-econ-figures/` in this repository.

## Explanatory diagrams

Three schematic diagrams explain what this base *is*, for a reader who needs the
construction rather than the results. They are hand-authored SVG in the citegraph
repository, in the house style of `docs/figures/pipeline.svg` — not matplotlib output
from this notebook, so they do not follow the journal grayscale conventions above:

- [`institutional_layers.svg`](../docs/figures/institutional_layers.svg) — the
  co-citation layer and the institutional layer over the same people, joined on
  `author_id`.
- [`institutional_build.svg`](../docs/figures/institutional_build.svg) — spreadsheet
  to crosswalks to the four tidy tables, with the ROR and three-valued-missing decisions.
- [`institutional_ties.svg`](../docs/figures/institutional_ties.svg) — what one
  supervision tie records, and how much of it is observed.

Each has a PDF twin beside it. Every count on them is real and was checked against
`institutional/out/` on 12 September 2026, after postdoc supervision was parked;
re-check them if the base changes.


## Setup

In [ ]:
import os
from pathlib import Path

import citegraph

# The replication root is the folder holding citegraph_out/ and institutional/;
# every path below hangs off it, so the package runs wherever it is cloned.
ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "citegraph_out").is_dir() and (p / "institutional").is_dir()
)
print(f"citegraph {citegraph.__version__} from {Path(citegraph.__file__).parent}")
print(f"replication root: {ROOT}")


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

# Every mark in this notebook comes from citegraph.plotting, which is the single
# place the economics-journal conventions are defined. The companion notebook
# paper4_figures.ipynb draws from the same module, so the two figure sets share
# one visual language by construction rather than by convention.
from citegraph.plotting import (
    BAR_FILL,
    GRAYS,
    INK,
    HATCHES,
    LABEL_OFFSET,
    NODE_RADIUS,
    RULE_COLOR,
    arrow_props,
    bar,
    barh,
    box_node,
    draw_node,
    flow_band,
    econ_style,
    figure_size,
    hairline,
    integer_ticks,
    junction_dot,
    node_legend,
    save_figure,
    value_labels,
)
from citegraph.plotting import show_caption as _show_caption

# Output of institutional/build_clean_base.py.
INST_DIR = ROOT / "institutional" / "out"

FIGURE_DIR = INST_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_NOTE = (
    "Hand-collected career metadata for the most prominent authors of the paper 4 core "
    "corpus, normalised with the crosswalks in paper4/institutional/."
)


def show_caption(caption, *, notes=None, source=SOURCE_NOTE, number=None):
    """Render the caption block under a figure, defaulting to this notebook's source."""
    _show_caption(caption, notes=notes, source=source, number=number)


authors = pd.read_csv(INST_DIR / "base_institucional_clean.csv")
positions = pd.read_csv(INST_DIR / "career_positions.csv")
supervision = pd.read_csv(INST_DIR / "supervision.csv")
flags = pd.read_csv(INST_DIR / "review_flags.csv")

print(f"{len(authors)} authors, {len(positions)} career positions, "
      f"{len(supervision)} supervision edges, {len(flags)} review flags")

### What the data does and does not support

Two properties of these tables decide how the figures below are built.

**Missing values are three-valued, not binary.** The source spreadsheet used `No`
(genuinely none), `Not found` (unknown), and blank interchangeably. The pipeline keeps them
apart, so any share must use the known cases as its denominator rather than all 58 authors.

**PhD supervision is recorded; postdoc supervision is not modelled at all.** The
spreadsheet has a `PhD_supervisor` column and no postdoc equivalent, and public sources
document postdoc advisors too patchily to be worth reconstructing. `supervision_seed.csv`
is therefore empty on purpose — seven researched ties are parked in
`supervision_seed.parked.csv` — and every edge in the network below is doctoral. Read it as
a near-complete PhD lineage that understates mentorship, not as a full supervision network.

In [ ]:
# Denominators, stated once so the figures below can refer back to them.
known_postdoc = authors["has_postdoc"].notna().sum()
did_postdoc = (authors["has_postdoc"] == True).sum()  # noqa: E712
n_postdoc_positions = (positions["position_type"] == "postdoc").sum()

print(f"postdoc known for {known_postdoc}/{len(authors)} authors; "
      f"{did_postdoc} did one ({did_postdoc / known_postdoc:.0%} of the known cases)")
print(f"{n_postdoc_positions} postdoc positions recorded; postdoc supervision is not modelled")
assert (supervision["relation_type"] == "phd").all(),     "non-PhD supervision ties present -- the figures below assume PhD-only"
print()
print(flags["kind"].value_counts().to_string())

## Figure 1. Field of current affiliation

Department strings are mapped onto fields by `crosswalk_fields.csv`. 58 authors hold 31
distinct department strings, so the raw strings are a list of near-singletons rather than
a distribution; the crosswalk is what makes them countable.

In [ ]:
field_counts = (
    authors.loc[authors["affiliation_field"].notna(), "affiliation_field"]
    .value_counts()
    .sort_values()
)

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.5))
    bars = barh(ax, field_counts.index, field_counts.to_numpy())
    value_labels(ax, bars, fmt="{:,.0f}")
    ax.set_xlabel("Authors")
    ax.set_xlim(0, field_counts.max() * 1.12)
    integer_ticks(ax, "x")
    fig.tight_layout()

    paths = save_figure(
        fig, FIGURE_DIR / "fig1_affiliation_field",
        caption="Field of current institutional affiliation",
        notes=(
            f"N = {int(field_counts.sum())} authors. Departments are mapped to fields by "
            "crosswalk_fields.csv, which tries its most specific rule first, so a "
            "department naming several units is assigned by the first rule that matches "
            "rather than by the unit named first. 'Non-academic / applied' covers positions "
            "at the ILO, a conservation NGO and a health-systems research centre."
        ),
    )
plt.show()
show_caption(
    "Field of current institutional affiliation",
    notes=(
        f"N = {int(field_counts.sum())} authors. Departments are mapped to fields by "
        "crosswalk_fields.csv, which assigns a department naming several units by the "
        "first rule that matches rather than by the unit named first."
    ),
)

## Figure 2. Country of current affiliation

Countries come from the ROR record of the canonical institution, not from the country
column of the spreadsheet, so a mis-keyed country cannot propagate into the count.

In [ ]:
country_counts = (
    positions.loc[positions["position_type"] == "affiliation", "country"]
    .dropna()
    .value_counts()
    .sort_values()
)

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.58))
    bars = barh(ax, country_counts.index, country_counts.to_numpy())
    value_labels(ax, bars, fmt="{:,.0f}")
    ax.set_xlabel("Authors")
    ax.set_xlim(0, country_counts.max() * 1.12)
    integer_ticks(ax, "x")
    fig.tight_layout()

    save_figure(
        fig, FIGURE_DIR / "fig2_affiliation_country",
        caption="Country of current institutional affiliation",
        notes=(
            f"N = {int(country_counts.sum())} authors. Country is taken from the ROR record "
            "of the canonical institution rather than from the country column of the source "
            "spreadsheet. One institution has no ROR record; for that row the spreadsheet's "
            "own country is used, spelled ROR's way so it cannot become a second category."
        ),
    )
plt.show()
show_caption(
    "Country of current institutional affiliation",
    notes=(
        f"N = {int(country_counts.sum())} authors. Country is taken from the ROR record of "
        "the canonical institution rather than from the source spreadsheet, except for the "
        "one institution that has no ROR record."
    ),
)

## Figure 3. Field of PhD

The same crosswalk applied to `PhD_title`. Compare with Figure 1: the gap between the two
is disciplinary movement between the doctorate and the current post.

In [ ]:
phd_counts = (
    authors.loc[authors["phd_field"].notna(), "phd_field"]
    .value_counts()
    .sort_values()
)

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.46))
    bars = barh(ax, phd_counts.index, phd_counts.to_numpy())
    value_labels(ax, bars, fmt="{:,.0f}")
    ax.set_xlabel("Authors")
    ax.set_xlim(0, phd_counts.max() * 1.12)
    integer_ticks(ax, "x")
    fig.tight_layout()

    save_figure(
        fig, FIGURE_DIR / "fig3_phd_field",
        caption="Field of doctorate",
        notes=(
            f"N = {int(phd_counts.sum())} authors with a recorded PhD title. Two authors have "
            "no doctorate and one row of the source carries no fields at all. A doctorate in "
            "progress is counted here at its field."
        ),
    )
plt.show()
show_caption(
    "Field of doctorate",
    notes=(
        f"N = {int(phd_counts.sum())} authors with a recorded PhD title. Two authors have no "
        "doctorate and one row of the source carries no fields at all."
    ),
)

In [ ]:
# Movement between the field of the doctorate and the field of the current post.
moved = authors[
    authors["phd_field"].notna()
    & authors["affiliation_field"].notna()
    & (authors["phd_field"] != authors["affiliation_field"])
]
both = authors[authors["phd_field"].notna() & authors["affiliation_field"].notna()]
print(f"{len(moved)}/{len(both)} authors sit in a different field from their doctorate")
print(
    moved.groupby(["phd_field", "affiliation_field"]).size()
    .sort_values(ascending=False).head(8).to_string()
)

## Figure 4. Field of doctorate by decade

54 doctorates spread over 1965–2022 is roughly one per year, so an annual series would be
noise. Decade bins give six groups; the earliest two are thin and the notes say so.

In [ ]:
by_decade = authors[authors["phd_decade"].notna() & authors["phd_field"].notna()]
table = (
    by_decade.pivot_table(index="phd_decade", columns="phd_field",
                          aggfunc="size", fill_value=0)
    .sort_index()
)
# Order fields by overall size so the stack reads consistently bottom-up.
table = table[table.sum().sort_values(ascending=False).index]
decade_n = table.sum(axis=1)

# GRAYS and HATCHES hold five entries each, so a plain `i % 5` on six fields
# hands the sixth the same fill AND the same hatch as the first. Pair them
# explicitly instead, varying shade and hatch together.
ENCODINGS = [
    (GRAYS[0], HATCHES[0]), (GRAYS[1], HATCHES[1]), (GRAYS[3], HATCHES[2]),
    (GRAYS[2], HATCHES[3]), (GRAYS[4], HATCHES[4]), (GRAYS[3], HATCHES[0]),
    (GRAYS[1], HATCHES[2]), (GRAYS[4], HATCHES[1]), (GRAYS[2], HATCHES[0]),
]
if len(table.columns) > len(ENCODINGS):
    raise ValueError(
        f"{len(table.columns)} fields but only {len(ENCODINGS)} distinct encodings; "
        "merge fields in crosswalk_fields.csv or extend ENCODINGS."
    )

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.55))
    bottom = pd.Series(0, index=table.index, dtype=float)
    for i, field in enumerate(table.columns):
        color, hatch = ENCODINGS[i]
        bar(
            ax, table.index, table[field], bottom=bottom,
            label=field, color=color, hatch=hatch,
        )
        bottom += table[field]
    ax.set_ylabel("Doctorates awarded")
    ax.set_xlabel("Decade of doctorate")
    integer_ticks(ax, "y")
    ax.legend(frameon=False, fontsize=7, loc="upper left", ncol=1)
    fig.tight_layout()

    save_figure(
        fig, FIGURE_DIR / "fig4_phd_field_by_decade",
        caption="Field of doctorate by decade awarded",
        notes=(
            f"N = {int(decade_n.sum())} doctorates with both a year and a field recorded. "
            "Counts per decade are "
            + ", ".join(f"{d}: {int(n)}" for d, n in decade_n.items())
            + ". The 1960s and 1980s bins hold few observations; read composition there "
            "with care. Fields are stacked largest-first from the axis."
        ),
    )
plt.show()
show_caption(
    "Field of doctorate by decade awarded",
    notes=(
        f"N = {int(decade_n.sum())} doctorates. Counts per decade are "
        + ", ".join(f"{d}: {int(n)}" for d, n in decade_n.items())
        + ". The earliest bins are thin; read composition there with care."
    ),
)

## The supervision network

`supervision.csv` is one edge table for both levels: each row is an advisor, an advisee, and
a `relation_type` of `phd`, `postdoc` or `visiting`. PhD edges come from the spreadsheet's
own column; postdoc edges come from `supervision_seed.csv` and are sparse.

Two views follow, because they answer different questions. The first shows *who is
connected to whom* and the second shows *when*, which a force-directed layout cannot.

In [ ]:
G = nx.DiGraph()
for e in supervision.itertuples(index=False):
    G.add_edge(e.advisor, e.advisee, relation=e.relation_type, confidence=e.confidence)

# Everyone in the 58-person list, whether or not they have an edge.
listed = set(authors["display_name"])
for name in sorted(listed):
    G.add_node(name)
nx.set_node_attributes(G, {n: (n in listed) for n in G}, "in_list")

phd_year = dict(zip(authors["display_name"], authors["phd_year_end"]))
phd_field = dict(zip(authors["display_name"], authors["phd_field"]))
nx.set_node_attributes(G, phd_year, "phd_year")

# Tie-break on a name: `key=len` alone leaves same-size components in hash order.
components = sorted(nx.connected_components(G.to_undirected()),
                    key=lambda c: (-len(c), min(c)))
print(f"{G.number_of_nodes()} people, {G.number_of_edges()} supervision ties")
print(f"{len(components)} components; largest holds {len(components[0])} people")
print(f"{sum(1 for n in G if G.degree(n) == 0)} people have no recorded tie")
print()
print("most connected:")
print(pd.Series(dict(G.degree())).sort_values(ascending=False).head(8).to_string())

### Drawing conventions for the two network figures

Both figures share these. Arrows run advisor → advisee, and every tie is doctoral
supervision. A node is filled when the person is one of the 59 core authors and open when
they are an advisor from outside that list.

Labels sit to the right of their node, which is why the helper below measures each label's
real width. An edge anchored at the node *centre* runs behind its own label, and the white
backing that used to hide it chopped every edge into stubs — the fragments that used to
appear beside "Samuel Bowles" and "Magnus Johannesson" were edges, not marks. Edges here
start at the right edge of the label and stop at the edge of the target node, so no
connector ever crosses a name.

Three traps worth recording:

- `econ_style` installs a property cycle that includes line styles, so a bare `ax.plot`
  silently comes out dashed or dotted. Every connector below pins `linestyle="-"`.
- `annotate`'s arrow patch scales its head by `mutation_scale`. Left at its default, at
  these data ranges, the arrowheads come out larger than the trees they point at.
- `networkx.draw_networkx_edges` calls `ax.tick_params(labelbottom=False)` internally,
  because it assumes a bare network diagram. Neither figure uses it any more — both route
  their own connectors — so that trap no longer applies here, but it will bite again the
  moment `nx.draw_*` is reintroduced on an axis that carries a real scale.

In [ ]:
# Layout primitives for the network figures below.
#
# The node marks, connectors, arrows and band rules all come from
# citegraph.plotting, so these figures share one vocabulary with the co-citation
# maps in paper4_figures.ipynb: a filled node is someone inside the population
# being described, an open node is someone outside it.
#
# What stays local is the *measurement* problem, which is specific to these
# figures. Both put a label to the right of its node, so an edge anchored at the
# node centre runs behind its own label -- that is what chopped the earlier edges
# into stubs beside "Samuel Bowles" and "Magnus Johannesson". Everything here
# anchors an edge at the right edge of the label instead, which means the layout
# needs each label's real width.
#
# Widths are measured off the renderer in typographic points, independent of the
# axis limits. Figure 5 then works in point space directly (the axes fills the
# figure and one data unit is one point); the figures on a year axis convert
# points to years with the axes width, which is known before any data is drawn.
from matplotlib.text import Text

FIG_W_IN = 6.5
AX_L, AX_W = 0.006, 0.988          # axes box for the year-axis figures
AX_PT = FIG_W_IN * 72.0 * AX_W     # its width in points
NODE_R, LABEL_OFF = NODE_RADIUS, LABEL_OFFSET
PAD = 2.8                          # clearance between a label edge and a connector
ARROW = arrow_props()


def text_pt(fig, s, fontsize):
    """Width of ``s`` in points at ``fontsize``, independent of the axis limits."""
    t = Text(0, 0, s, fontsize=fontsize)
    t.set_figure(fig)
    return t.get_window_extent(renderer=fig.canvas.get_renderer()).width / fig.dpi * 72.0


def measure(names, fontsize):
    """Label widths in points, measured on a throwaway figure of the final width."""
    probe = plt.figure(figsize=(FIG_W_IN, 4))
    probe.canvas.draw()
    try:
        return {n: text_pt(probe, n, fontsize) for n in names}
    finally:
        plt.close(probe)


def author_node(ax, x, y, name, *, fontsize, unit=1.0):
    """A person in a supervision figure: filled if they are one of the listed
    authors, open if they are an advisor from outside the list."""
    return draw_node(ax, x, y, name, inside=name in listed,
                     fontsize=fontsize, unit=unit, zorder=4)


def author_legend(ax, loc, fontsize, **kw):
    return node_legend(ax, "In the core author list", "Advisor outside the list",
                       loc=loc, fontsize=fontsize, **kw)


def longest_path_layers(H):
    """Generation of each node = longest path from any source."""
    layer = {n: 0 for n in H}
    for n in nx.topological_sort(H):
        for s in H.successors(n):
            layer[s] = max(layer[s], layer[n] + 1)
    layers = {}
    for n, L in layer.items():
        layers.setdefault(L, []).append(n)
    for L in layers:
        layers[L].sort()
    return layer, layers


def barycentre_order(H, layers, sweeps=8):
    """Order nodes inside each generation so the brackets cross as little as possible."""
    for _ in range(sweeps):
        for L in sorted(layers)[1:]:
            prev = {n: i for i, n in enumerate(layers[L - 1])}
            layers[L].sort(key=lambda n: (
                sum(prev.get(p, 0) for p in H.predecessors(n) if p in prev)
                / max(1, sum(1 for p in H.predecessors(n) if p in prev)), n))
        for L in sorted(layers, reverse=True)[1:]:
            nxt = {n: i for i, n in enumerate(layers.get(L + 1, []))}
            layers[L].sort(key=lambda n: (
                sum(nxt.get(c, 0) for c in H.successors(n) if c in nxt)
                / max(1, sum(1 for c in H.successors(n) if c in nxt)), n))
    return layers

### Figure 5. Doctoral lineages

With postdoc ties out of scope the network is a forest of small trees — see the component
sizes printed above; the largest holds nine people. "The largest connected component" would
therefore throw away most of the structure, so this draws every lineage of at least five
people, stacked.

It is drawn as a dendrogram: generations are columns, each column is exactly as wide as its
longest label plus a routing channel, and every bracket routes through a lane of its own in
that channel. That is why a force-directed layout was unusable when it was tried first — these
are near-trees, and the springs pushed them into overlapping clumps.

Near-trees, not trees, and the difference is visible in the top band. A doctorate can have
several supervisors — Anna Dreber's has three — so a bracket sometimes has to span rows that
other brackets are using, and some crossings are unavoidable. Two conventions make them
readable rather than misleading: lanes are spaced far enough apart to read as separate lines,
and **a dot marks a point where connectors meet, so lines that merely cross are bare**. Follow
Martin A. Nowak's line and it ends at the arrow into Dreber; it passes Juan Camilo Cárdenas's
row without a dot, because Cárdenas is Samuel Bowles's student and no one else's.

Filled nodes are authors in the core list; open nodes are advisors who are not themselves in
it. The open nodes in generation 1 are where this community's training came from.

In [ ]:
# Figure 5. Doctoral lineages, drawn as a tidy tree.
#
# The forest is drawn as a dendrogram: generations are columns, and every column
# is exactly as wide as its widest node plus a routing channel, so the vertical
# leg of a bracket always lands in empty space. That is what lets the whole
# figure be crossing-free, and it is why a force-directed layout was never going
# to work here -- these are trees, and the springs pushed them into clumps.
#
# The router is written against a *node shape* rather than against a label,
# because figure 8 further down draws this same forest with boxed two-line
# nodes. A shape reports the four things the layout actually needs -- how wide a
# node is drawn from its column origin, how far left of that origin an incoming
# arrow should stop, how tall a row is, and how to draw one -- so the three tree
# figures share one router instead of forking it.
#
# Crossings cannot be designed away here, and an earlier version of this cell
# claimed otherwise. A forest routes crossing-free because each node has one
# parent and therefore one bracket owning its rows; this graph is a DAG, because
# a doctorate can be supervised by three people. Anna Dreber's is, so her bracket
# has to span four rows of generation 1 to reach Johannesson, Nowak and Bowles --
# and Bowles sits on Juan Camilo Cardenas's row, because Cardenas is his only
# student. Something has to cross. What the router owes the reader is therefore
# not the absence of crossings but the means to tell one from a junction:
# lanes far enough apart to be separate lines, and a dot wherever connectors
# actually meet. Without both, the figure read as Nowak having trained Cardenas.

# Every tree figure carries this, because each is read on its own page and the
# dot is doing real work: without it a reader has no way to tell a bracket that
# stops at a row from one passing through it.
ROUTING_NOTE = (
    "Arrows run advisor to advisee; all ties are doctoral. A dot marks a point "
    "where connectors meet -- lines that cross without one are unconnected."
)

CHANNEL_MIN = 20.0   # narrowest a gutter ever gets, whatever it has to carry
LANE_GAP = 6.0       # separation between one bracket's lane and the next
LANE_NEAR = 7.0      # column origin to the lane nearest it, clearing the arrow
LANE_PAD = 3.0       # clearance left of the outermost lane


def dot_shape(nodes, fs):
    """A node as figure 5 draws it: a marker with its name to the right."""
    w = measure(nodes, fs)
    return {
        "node_w": {n: LABEL_OFF + w[n] for n in nodes},
        "tip_gap": NODE_R + 0.6,
        "start": NODE_R + 1.0,
        "head_dx": -NODE_R,
        "row": fs * 1.80,
        "pad_top": 2.0,
        "pad_bottom": 2.4,
        "fs": fs,
        "draw": lambda ax, x0, y, n: author_node(ax, x0, y, n, fontsize=fs),
        "legend": lambda ax: author_legend(ax, "lower left", fs - 0.4,
                                           bbox_to_anchor=(0.0, 0.0)),
    }


def _rel_rows(layers):
    """Row offsets within one lineage, in units of one row.

    Each generation is centred on the band, so the band's own position cancels
    and this is exactly the vertical order ``lineage_tree`` will lay out --
    available before any figure exists, which is what lets the gutters be sized
    from it.
    """
    return {n: (len(ns) - 1) / 2 - i
            for ns in layers.values() for i, n in enumerate(ns)}


def _brackets(H, layer, rows):
    """Advisee -> (advisors, whether the bracket has to turn a corner)."""
    by_target = {}
    for u, v in sorted(H.edges()):
        by_target.setdefault(v, []).append(u)
    reach = {t: [rows[t]] + [rows[s] for s in ss] for t, ss in by_target.items()}
    return by_target, reach


def _lane_demand(trees):
    """How many bracket lanes each generation's gutter has to hold at once.

    Only a bracket that turns a corner needs a lane. An advisee with a single
    advisor already on its own row -- Cardenas under Bowles -- is one straight
    line from name to name, and reserving channel width for it would squeeze the
    brackets that do need it: generation 2 of the Swedish band has four advisees
    but only two of them bend. Sized per generation for the same reason.
    """
    need = {}
    for H, layer, layers in trees:
        rows = _rel_rows(layers)
        _, reach = _brackets(H, layer, rows)
        per_layer = {}
        for t, ys in reach.items():
            if max(ys) - min(ys) > 1e-9:
                per_layer[layer[t]] = per_layer.get(layer[t], 0) + 1
        for L, n in per_layer.items():
            need[L] = max(need.get(L, 0), n)
    return need


def _columns(lineages, shape):
    """Per-generation x positions, each column as wide as its widest node and
    each gutter as wide as the brackets that have to route through it."""
    trees = []
    for comp in lineages:
        H = G.subgraph(comp).copy()
        layer, layers = longest_path_layers(H)
        trees.append((H, layer, barycentre_order(H, layers)))
    node_w = shape["node_w"]
    content = {}
    for _, _, layers in trees:
        for L, ns in layers.items():
            content[L] = max(content.get(L, 0.0), max(node_w[n] for n in ns))
    need = _lane_demand(trees)
    channel = {
        L: max(CHANNEL_MIN, LANE_NEAR + max(0, need.get(L, 1) - 1) * LANE_GAP + LANE_PAD)
        for L in content
    }
    x, cur = {}, shape["start"]
    for L in sorted(content):
        x[L] = cur
        cur += content[L] + channel.get(L + 1, CHANNEL_MIN)
    last = max(content)
    return trees, x, channel, x[last] + content[last] + 2.0


def _fit_fontsize(lineages, make_shape, lo=6.4, hi=9.0):
    """The largest type that still fits the printed text width."""
    nodes = {n for comp in lineages for n in comp}
    best = lo
    for _ in range(24):
        mid = (lo + hi) / 2
        if _columns(lineages, make_shape(nodes, mid))[3] <= FIG_W_IN * 72.0 - 6:
            best, lo = mid, mid
        else:
            hi = mid
    return best


def _detour_y(y_t, blocked, row):
    """A clear horizontal lane for a generation-skipping edge, as near the target
    as possible. Such an edge crosses whole columns, so it has to miss every label
    in them -- landing half a row off the target put one through Vélez's name."""
    for k in range(13):
        for dy in ({y_t} if k == 0 else {y_t + k * row / 4, y_t - k * row / 4}):
            if all(abs(dy - o) >= row * 0.42 for o in blocked):
                return dy
    return y_t + row


def _junction_dots(ax, segments, tol=0.05):
    """Dot every point where three or more connector directions meet.

    Only segment *endpoints* are candidates, which is the whole point: a line
    that runs through a point without ending there is crossing over, and gets no
    dot. So the reader can tell Bowles's line forking to two students from
    Dreber's spine passing Cardenas's row on its way to Nowak.

    Corners and straight joins are left bare. Three directions is the threshold
    because a dot on every bend is a dot that means nothing.
    """
    hs = [s for s in segments if s[0] == "h"]
    vs = [s for s in segments if s[0] == "v"]
    ends = {(round(x0, 3), round(yy, 3)) for _, yy, x0, _ in hs}
    ends |= {(round(x1, 3), round(yy, 3)) for _, yy, _, x1 in hs}
    ends |= {(round(xx, 3), round(y0, 3)) for _, xx, y0, _ in vs}
    ends |= {(round(xx, 3), round(y1, 3)) for _, xx, _, y1 in vs}
    for px, py in sorted(ends):
        legs = (
            any(abs(yy - py) <= tol and x0 < px - tol <= x1 for _, yy, x0, x1 in hs)
            + any(abs(yy - py) <= tol and x0 <= px + tol < x1 for _, yy, x0, x1 in hs)
            + any(abs(xx - px) <= tol and y0 < py - tol <= y1 for _, xx, y0, y1 in vs)
            + any(abs(xx - px) <= tol and y0 <= py + tol < y1 for _, xx, y0, y1 in vs)
        )
        if legs >= 3:
            junction_dot(ax, px, py)


def _route(ax, x, channel, y, shape, layer, H):
    """Bracket routing: source right edge -> the bracket's own lane -> target.

    Each bracket that bends owns a lane in the gutter left of its column, and the
    lanes are ordered by vertical reach, the **shortest nearest its own column**.
    The ordering is load-bearing and the other way round is worse, which is only
    obvious once you notice that every arrowhead is itself a horizontal line
    needing to cross the gutter. Put the band-spanning spine nearest the column
    and it runs the length of the generation a few points from every node, with
    all four arrows striking it on their way in -- which is precisely the picture
    that reads as one trunk feeding everybody. Put it furthest out and the arrows
    are clear of it; what crosses it instead is the occasional advisor's leg,
    out in the empty gutter where a bare crossing is legible as a crossing.
    """
    node_w, row = shape["node_w"], shape["row"]
    rows = {n: y[n] for n in y}
    by_target, reach = _brackets(H, layer, rows)
    per_layer = {}
    for t in by_target:
        if max(reach[t]) - min(reach[t]) > 1e-9:
            per_layer.setdefault(layer[t], []).append(t)
    for L in per_layer:
        per_layer[L].sort(key=lambda n: (max(reach[n]) - min(reach[n]), n))
    # A straight-line bracket keeps the nearest position without consuming it:
    # it draws no spine, so nothing can collide with it there.
    slot = {t: i for ts in per_layer.values() for i, t in enumerate(ts)}

    segments = []

    def h(x0, x1, yy):
        hairline(ax, [x0, x1], [yy, yy])
        segments.append(("h", yy, min(x0, x1), max(x0, x1)))

    def v(xx, y0, y1):
        if abs(y1 - y0) > 1e-9:
            hairline(ax, [xx, xx], [y0, y1])
        segments.append(("v", xx, min(y0, y1), max(y0, y1)))

    for t, srcs in by_target.items():
        cx = x[layer[t]] - LANE_NEAR - slot.get(t, 0) * LANE_GAP
        ys = [y[t]]
        for s in srcs:
            x0 = x[layer[s]] + node_w[s] + PAD
            if layer[t] - layer[s] == 1:
                h(x0, cx, y[s])
                ys.append(y[s])
            else:
                cs = (x[layer[s]] + node_w[s]
                      + channel.get(layer[s] + 1, CHANNEL_MIN) * 0.45)
                dy = _detour_y(y[t], [y[n] for n in y if layer[s] < layer[n] < layer[t]], row)
                h(x0, cs, y[s])
                v(cs, y[s], dy)
                h(cs, cx, dy)
                ys.append(dy)
        v(cx, min(ys), max(ys))
        tip = x[layer[t]] - shape["tip_gap"]
        ax.annotate("", xy=(tip, y[t]), xytext=(cx, y[t]), arrowprops=ARROW, zorder=3)
        segments.append(("h", y[t], min(cx, tip), max(cx, tip)))

    _junction_dots(ax, segments)


def lineage_tree(min_lineage, stem, *, keep=None, make_shape=dot_shape, caption=None,
                 notes=None, show=True):
    """Draw every lineage of at least ``min_lineage`` people, stacked.

    ``keep`` is an optional predicate on a lineage -- the set of names in one
    connected component -- applied after the size threshold, so one call can draw
    a subset of the forest without the caller having to reach into the layout.
    """
    lineages = [c for c in components if len(c) >= min_lineage]
    if keep is not None:
        lineages = [c for c in lineages if keep(c)]
    shown = sum(len(c) for c in lineages)
    fs = _fit_fontsize(lineages, make_shape)
    shape = make_shape({n for comp in lineages for n in comp}, fs)
    trees, x, channel, total = _columns(lineages, shape)

    row = shape["row"]
    pos, rules, cursor = {}, [], 0.0
    for _, layer, layers in trees:
        height = (max(len(ns) for ns in layers.values()) - 1) * row
        centre = cursor - height / 2
        for L, ns in layers.items():
            for i, n in enumerate(ns):
                pos[n] = (x[L], centre + ((len(ns) - 1) / 2 - i) * row)
        cursor -= height + row * 1.35
        rules.append(cursor + row * 0.675)
    rules.pop()

    ys = [p[1] for p in pos.values()]
    top = max(ys) + row * shape["pad_top"]
    bottom = min(ys) - row * shape["pad_bottom"]

    with econ_style():
        fig = plt.figure(figsize=figure_size("text", height=(top - bottom) / 72.0))
        ax = fig.add_axes([0, 0, 1, 1])
        ax.set_xlim(0, FIG_W_IN * 72.0)
        ax.set_ylim(bottom, top)
        ax.axis("off")

        for L in sorted(x):
            ax.text(x[L] + shape["head_dx"], max(ys) + row * 0.95, f"Generation {L + 1}",
                    fontsize=fs - 0.9, color="#444444", ha="left", va="baseline")
        for r in rules:
            hairline(ax, [0, total], [r, r], color=RULE_COLOR, linewidth=0.4, zorder=0)
        for H, layer, _ in trees:
            _route(ax, x, channel, {n: pos[n][1] for n in H}, shape, layer, H)
        for n, (nx_, ny) in pos.items():
            shape["draw"](ax, nx_, ny, n)
        shape["legend"](ax)

        if caption is None:
            caption = "Doctoral lineages with at least %d people" % min_lineage
        if notes is None:
            # Say only what is true of *this* selection: once `keep` is in play,
            # the omitted lineages are no longer all omitted for being small.
            why = (f"hold fewer than {min_lineage} people each"
                   if keep is None else
                   f"either hold fewer than {min_lineage} people or fall outside "
                   f"what this panel selects")
            notes = (
                f"{shown} of {G.number_of_nodes()} people in {len(lineages)} lineages. The "
                f"supervision network has {len(components)} components in all; the "
                f"{len(components) - len(lineages)} omitted ones {why}. Horizontal position "
                f"is generation, not year. {ROUTING_NOTE}"
            )
        save_figure(fig, FIGURE_DIR / stem, caption=caption, notes=notes)
    if show:
        plt.show()
        show_caption(caption, notes=notes)
    else:
        plt.close(fig)
    print(f"{stem}: {len(lineages)} lineages, {shown} people, {fs:.1f} pt type, "
          f"{total:.0f}/{FIG_W_IN * 72:.0f} pt wide, "
          f"{(top - bottom) / 72:.2f} in tall")


lineage_tree(5, "fig5_doctoral_lineages")


#### Two alternatives to weigh against it

Neither is written to the canonical `fig5_doctoral_lineages` stem, so the manuscript keeps
pointing at the tree above until one of these is chosen to replace it.

The first puts the same six lineages on a real year axis. It is worth seeing, but it is
weaker here for a structural reason rather than a cosmetic one: a year axis has no
label-free channels for connectors to route through, so the brackets become diagonals, and
seventeen of these thirty-seven people are advisors outside the list with no recorded
doctorate year. They go in a gutter rather than at an invented position, which is honest but
spends a third of the width on "unknown".

The second drops the threshold from five people to three and then splits the result in two,
by how many of the 58 listed authors a lineage contains. The threshold alone raises coverage
from 37 to 64 people but produces a full-page figure whose bottom half is a run of
near-identical two-generation pairs; the split is what fixes that, because the repetition sits
entirely on one side. Every lineage with a single listed author has the same shape — that
author and the two people who supervised them — so they collect into one compact panel, while
the lineages with two or more are chains the community produced itself. The pair therefore
answers two questions instead of blurring them: where this community's training came from, and
where it reproduced.

In [ ]:
# Alternative A: the same six lineages placed on a real year axis.
#
# Worth looking at, but weaker than the tree above for this data, and the reason
# is structural rather than cosmetic. A year axis has no label-free channels for
# edges to route through, so connectors have to be diagonals; and 17 of these 37
# people are advisors outside the 58-author list, for whom no doctorate year was
# ever collected. They go in a left gutter rather than being given an invented
# position, which is honest but spends a third of the width on "unknown".

def lineages_in_time(min_lineage, stem, *, fs=8.0, show=True):
    lineages = [c for c in components if len(c) >= min_lineage]
    nodes = [n for c in lineages for n in c]
    H = G.subgraph(nodes)
    year = {n: (None if pd.isna(phd_year.get(n, float("nan"))) else float(phd_year[n]))
            for n in nodes}
    dated = {n for n in nodes if year[n] is not None}
    undated = [n for n in nodes if n not in dated]
    wpt = measure(nodes, fs)

    gutter_pt = max(wpt[n] for n in undated) + LABEL_OFF + 2 * NODE_R + 6.0
    lo, hi = min(year[n] for n in dated), max(year[n] for n in dated)
    span = (hi - lo) * 2.0                 # solve for a span that fits gutter + labels
    for _ in range(60):
        need = max(year[n] + (LABEL_OFF + wpt[n] + 4) / AX_PT * span
                   - (lo - 3 + span - gutter_pt / AX_PT * span) for n in dated)
        if need <= 0:
            break
        span += need * 1.05
    unit = span / AX_PT                    # data units per point
    sep = lo - 3.0                         # gutter / year-axis boundary
    label_right = sep - 3.0 * unit
    xmin = sep - gutter_pt * unit

    row = fs * 1.85
    pos, rules, cursor = {}, [], 0.0
    for comp in lineages:
        # One row per dated person: sharing a row lets a long connector run straight
        # through a neighbour's name, and this axis cannot channel around it.
        dat = sorted((n for n in comp if n in dated), key=lambda n: (year[n], n))
        rows = {n: i for i, n in enumerate(dat)}
        mean_row = lambda n: (sum(rows.get(c, 0) for c in H.successors(n))
                              / max(1, sum(1 for c in H.successors(n) if c in rows)))
        used = set()
        for n in sorted((n for n in comp if n in undated), key=lambda n: (mean_row(n), n)):
            want = round(mean_row(n))
            r = next(k for k in sorted(range(len(comp)), key=lambda k: (abs(k - want), k))
                     if k not in used)
            used.add(r)
            rows[n] = r
        for n, r in rows.items():
            x0 = label_right - (LABEL_OFF + wpt[n]) * unit if n in undated else year[n]
            pos[n] = (x0, cursor - r * row)
        cursor -= (max(rows.values()) + 1) * row + row * 0.75
        rules.append(cursor + row * 0.375)
    rules.pop()

    ys = [p[1] for p in pos.values()]
    top, bottom = max(ys) + row * 1.5, min(ys) - row * 2.0
    axis_pt = 30.0
    fig_h_pt = (top - bottom) + axis_pt + 18.0

    with econ_style():
        fig = plt.figure(figsize=figure_size("text", height=fig_h_pt / 72.0))
        ax = fig.add_axes([AX_L, axis_pt / fig_h_pt, AX_W, (top - bottom) / fig_h_pt])
        ax.set_xlim(xmin, xmin + span)
        ax.set_ylim(bottom, top)
        # The gutter divider does a different job from the band rules, so it carries
        # the same rule colour at a heavier weight rather than a shade of its own.
        hairline(ax, [sep, sep], [bottom, max(ys) + row * 0.8],
                 color=RULE_COLOR, linewidth=0.8, zorder=0)
        ax.text(sep - 2.0 * unit, max(ys) + row * 1.05, "Doctorate year not recorded",
                fontsize=fs - 1.1, color="#444444", ha="right", va="baseline")
        for r in rules:
            hairline(ax, [xmin, xmin + span], [r, r], color=RULE_COLOR, linewidth=0.4, zorder=0)

        for u, v in sorted(H.edges()):
            xu, yu = pos[u]
            xv, yv = pos[v]
            x0 = (label_right + PAD * unit) if u in undated else xu + (LABEL_OFF + wpt[u] + PAD) * unit
            tip = xv - (NODE_R + 0.6) * unit
            if x0 < tip - 1.0:
                ax.annotate("", xy=(tip, yv), xytext=(x0, yu), arrowprops=ARROW, zorder=2)
            else:   # the advisor's own label overhangs the advisee's year
                hairline(ax, [xu, xu], [yu, yv])
                ax.annotate("", xy=(tip, yv), xytext=(xu, yv), arrowprops=ARROW, zorder=2)
        for n, (x, y) in pos.items():
            author_node(ax, x, y, n, fontsize=fs, unit=unit)

        ax.set_yticks([])
        ax.set_xticks([y for y in range(1970, 2021, 10) if y >= sep])
        ax.set_xlabel("Year of doctorate")
        for side in ("left", "right", "top"):
            ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_bounds(lo - 2, hi + 2)
        author_legend(ax, "lower right", fs - 0.4)
        caption = "Doctoral lineages placed by year of doctorate"
        notes = (f"The same {len(lineages)} lineages as the previous figure, each person at the "
                 f"year of their own doctorate. {len(undated)} of {len(nodes)} people -- almost "
                 "all advisors outside the 58-author list -- have no recorded doctorate year "
                 "and sit in the left-hand gutter rather than at an inferred position.")
        save_figure(fig, FIGURE_DIR / stem, caption=caption, notes=notes)
    if show:
        plt.show()
        show_caption(caption, notes=notes)
    else:
        plt.close(fig)
    print(f"{stem}: {len(nodes)} people, {len(undated)} with no recorded year")


lineages_in_time(5, "fig5alt_lineages_in_time")

# Alternative B: the threshold dropped to three people, and then split in two by
# how many of the 58 listed authors a lineage actually contains.
#
# Dropping the threshold alone took coverage from 37 to 64 people but produced a
# full-page figure whose bottom half was a run of near-identical two-generation
# pairs. Splitting is what fixes that, because the two halves are not the same
# kind of object and the repetition was entirely on one side: every lineage with
# a single listed author turns out to have exactly the same shape -- one author
# and the two people who supervised them -- while every lineage with two or more
# is a chain the community produced itself. Read as a pair they answer different
# questions: where this community's training came from, and where it reproduced.
_CORE = lambda comp: len(comp & listed)  # noqa: E731

# The single-author lineages are drawn top-down further below, with lineage_topdown.


### Figure 8. The same lineages, carrying each person's institution

Figure 5 answers *who trained whom*; this answers *where*, by putting the institution
inside the node the way the Victoria Chick network in the Review of Political Economy
does. It is the same forest through the same router, so the two can be read against each
other line for line.

The box is the only change. Fill still carries one bit and one bit only — filled for a
member of the author list, open for an advisor from outside it — and the institution is
text, never a second colour.


In [ ]:
# Figure 8. The same lineages, with each person's institution in the box.
#
# Figure 5 shows who trained whom. It cannot show where, and "where" is most of
# what a training network is for: a lineage rooted at Amherst and Gothenburg
# that now sits in Bogota and Medellin is a different object from one that never
# left its department. Putting the institution in the box is the move the
# Review of Political Economy's Victoria Chick network makes, and it is the only
# thing that changes here -- same lineages, same router, same fills.

# For someone in the author list the institution is their current affiliation:
# the fact this figure exists to show, and recorded for all 58. An advisor from
# outside the list has no career record at all, so the only institution we hold
# for them is the one on the tie itself -- where the doctorate they supervised
# was awarded. Two advisors supervised at more than one place; each carries the
# institution they supervised at most often, earliest year breaking a tie, and
# the caption says so rather than letting a box quietly drop the second one.
affiliation_short = dict(
    zip(authors["display_name"], authors["affiliation_institution_short"])
)


def _tie_institution(person):
    ties = supervision[supervision["advisor"] == person]
    if not len(ties):
        return ""
    ranked = (ties.groupby("institution_short")["year_end"].agg(["size", "min"])
              .sort_values(["size", "min"], ascending=[False, True]))
    return ranked.index[0]


inst_label = {
    n: (affiliation_short.get(n, "") if n in listed else _tie_institution(n))
    for n in sorted(G.nodes())
}
multi_site = sorted(
    p for p in G
    if p not in listed and supervision[supervision["advisor"] == p]["institution_short"].nunique() > 1
)
print(f"{sum(1 for v in inst_label.values() if v)} of {len(inst_label)} people carry "
      f"an institution; {len(multi_site)} advisors supervised at more than one: "
      f"{', '.join(multi_site)}")

# All three are in units of the type size, and they are what decides whether
# thirty-seven two-line boxes fit a page. At the airier settings a name box wants
# 3.55 type sizes of vertical room and the figure came out 10.05 in tall, which a
# letter-page text block cannot take without a reduction that drops the
# institution line below eight point. A box separates itself from its neighbour
# with its own edge, so the gap between boxes buys far less legibility than the
# leading inside one: shrinking the gap and the outer margins, and leaving the
# two lines a full type size apart, fits the same thirty-seven people.
BOX_PAD_X = 4.0    # clearance between a box edge and the text inside it
MIN_PT = 7.0       # smallest type at print size, institution lines included
BOX_GAP = 0.45     # gap between stacked boxes


def box_shape(nodes, fs):
    """A node as figure 8 draws it: a box carrying a name and an institution."""
    sub_fs = max(fs - 1.1, MIN_PT)
    head = measure(nodes, fs)
    sub = measure({inst_label[n] for n in nodes}, sub_fs)
    width = {n: max(head[n], sub[inst_label[n]]) + 2 * BOX_PAD_X for n in nodes}
    height = fs * 2.7

    def draw(ax, x0, y, n):
        return box_node(ax, x0, y, [n, inst_label[n]], width=width[n], height=height,
                        inside=n in listed, fontsize=fs, sub_fontsize=sub_fs)

    return {
        # A box is anchored on its own left edge, so the column origin is the
        # edge itself and an incoming arrow stops there rather than short of a
        # marker centre.
        "node_w": width,
        "tip_gap": 0.0,
        "start": 1.0,
        "head_dx": 0.0,
        "row": height + fs * BOX_GAP,
        "pad_top": 0.8,
        "pad_bottom": 1.6,
        "fs": fs,
        "draw": draw,
        # The second line means something different on each fill -- an
        # affiliation on one, a place of supervision on the other -- so the key
        # that explains the fill explains that too. Leaving it to the caption
        # invites the reader to take "Martin A. Nowak / Stockholm Univ." for an
        # affiliation, which it is not.
        "legend": lambda ax: node_legend(
            ax, "In the author list: current affiliation",
            "Advisor outside it: where they supervised",
            loc="lower left", fontsize=fs - 0.4, bbox_to_anchor=(0.0, 0.0), shape="box"),
    }


FIG8_MIN = 5

_shown = [c for c in components if len(c) >= FIG8_MIN]
_n = sum(len(c) for c in _shown)
lineage_tree(
    FIG8_MIN,
    "fig8_lineages_annotated",
    make_shape=box_shape,
    caption="Doctoral lineages, with each person's institution",
    notes=(
        f"The same {len(_shown)} lineages and {_n} people as the tree above. For a member of "
        f"the author list the second line is their current affiliation; for an advisor from "
        f"outside it, where the doctorate they supervised was awarded, which is the only "
        f"institution recorded for them. {' and '.join(multi_site)} each supervised at more than "
        f"one institution and carry the one they supervised at most often. Institution names "
        f"are shortened by institution_short_names.csv. Horizontal position is generation, "
        f"not year. {ROUTING_NOTE}"
    ),
)


#### Lineages linking two or more listed authors, one tree each

The nine lineages that hold at least three people and two or more of the 58 listed authors, each drawn as its own top-down tree -- except the three that are a single chain, which share the last figure: generation 1 at the top, cascading down. Each box gives the person's name and, under it, the institution and year of their doctorate where the base records one -- which is every listed author and none of the advisors from outside the list.


In [ ]:
# Alternative C: the lineages linking two or more listed authors, one top-down
# tree per lineage.
#
# Each lineage is its own figure, generation 1 at the top, cascading down, with
# each person's doctorate -- institution and year -- under their name. That
# second line is the advisee's own record, so it exists for every listed author
# and for nobody outside the list: the base never collected an outside advisor's
# doctorate, and an empty line is more honest than a guessed one.
#
# The reader's one question is "who supervised whom?", so every rule below is
# there to make each tie traceable on its own:
#
# * One line per tie, straight from the advisor's bottom edge to the student's
#   top edge (S-curves were tried and rejected on looks). The first version
#   shared horizontal buses between ties, the way the
#   left-to-right router does; with several supervisors per doctorate (Dreber has
#   three, Pfaff five) a bus carries several ties at once, and the reader had to
#   decode junction dots to tell who supervised Eva Ranehill. A line that is
#   exactly one tie needs no decoding.
# * Ports are spread along the box edge, ordered by where the other end sits, so
#   an advisor's three students leave as three separate lines that cannot cross
#   each other, and three advisors arrive as three separate arrowheads.
# * Order within a row is the exact crossing minimum, found by trying every order
#   (the largest lineage has 4! x 4! of them), compactness breaking ties. The
#   barycentre heuristic left four avoidable crossings in the Stockholm lineage.
# * A root sits one row above its highest student, not on row 1. Corbera
#   supervised only Lina Moros, a generation-3 doctorate; on row 1 his line would
#   have to run past Velez's row. Dropped a row, every tie spans exactly one gap.
import itertools
import math

# The full name of the university, not figure 8's short label: a tree has room
# for it, and "East Anglia" or "UNAL" tells a reader nothing. An acronym stays
# only where it is better known than the name it stands for.
KNOWN_ACRONYMS = {
    "Massachusetts Institute of Technology": "MIT",
    "London School of Economics and Political Science": "LSE",
    "University of California, Los Angeles": "UCLA",
}
phd_inst = {
    n: KNOWN_ACRONYMS.get(i, i) for n, i in {
        **dict(zip(supervision["advisee"], supervision["institution"])),
        **dict(zip(authors["display_name"], authors["phd_institution"])),
    }.items()
}
phd_end = {
    **dict(zip(supervision["advisee"], supervision["year_end"])),
    **dict(zip(authors["display_name"], authors["phd_year_end"])),
}


def phd_label(n):
    inst, year = phd_inst.get(n), phd_end.get(n)
    parts = [str(inst) if pd.notna(inst) else "", f"{year:.0f}" if pd.notna(year) else ""]
    return ", ".join(p for p in parts if p)


TD_GAP_X = 12.0      # horizontal space between neighbouring boxes
TD_GAP_Y = 26.0      # least vertical gap between generations
TD_SLOPE = 0.22      # ...grown so the most sideways tie stays visibly diagonal
TD_PORT = 11.0       # spacing of the ports along one box edge
TD_ORDER_BUDGET = 200_000

TD_NOTE = ("Each line is one doctoral tie, running from the advisor above to the "
           "student below.")


def _td_crossings(H, layers):
    Ls = sorted(layers)
    total = 0
    for a, b in zip(Ls, Ls[1:]):
        pa = {n: i for i, n in enumerate(layers[a])}
        pb = {n: i for i, n in enumerate(layers[b])}
        es = [(pa[u], pb[v]) for u, v in H.edges() if u in pa and v in pb]
        total += sum(1 for i, (u1, v1) in enumerate(es) for u2, v2 in es[i + 1:]
                     if (u1 - u2) * (v1 - v2) < 0)
    return total


def _td_spread(H, layers):
    """How far ties lean sideways, in row positions measured from each row's centre."""
    rel = {n: i - (len(ns) - 1) / 2 for ns in layers.values() for i, n in enumerate(ns)}
    return sum(abs(rel[u] - rel[v]) for u, v in H.edges())


def _td_order(H, layers):
    """The order of every row with fewest crossings, then least lean.

    Exhaustive, because these lineages are small enough that the exact answer is
    cheap; a heuristic ordering is what produced the crossings in the first place.
    Rows are enumerated from alphabetical order, so a tie resolves the same way
    on every run.
    """
    Ls = sorted(layers)
    n_orders = 1
    for L in Ls:
        n_orders *= math.factorial(len(layers[L]))
    if n_orders > TD_ORDER_BUDGET:
        return barycentre_order(H, layers)
    best, best_key = None, None
    for combo in itertools.product(*(itertools.permutations(layers[L]) for L in Ls)):
        cand = {L: list(c) for L, c in zip(Ls, combo)}
        key = (_td_crossings(H, cand), _td_spread(H, cand))
        if best_key is None or key < best_key:
            best, best_key = cand, key
    return best


def _td_layers(H):
    layer, _ = longest_path_layers(H)
    for n in H:
        if H.in_degree(n) == 0 and H.out_degree(n):
            layer[n] = min(layer[c] for c in H.successors(n)) - 1
    layers = {}
    for n in sorted(H):
        layers.setdefault(layer[n], []).append(n)
    return layer, _td_order(H, layers)


def _td_x(H, layers, width):
    """Centre x of every box: neighbours pulled together, order and gaps kept."""
    cx = {}
    for ns in layers.values():
        cur = 0.0
        for n in ns:
            cx[n] = cur + width[n] / 2
            cur += width[n] + TD_GAP_X

    def settle(ns, want):
        # Place at the wanted centres, then push right and back left so no two
        # boxes come closer than TD_GAP_X; the mean of the two passes keeps the
        # row centred on what it wanted.
        lo = []
        for i, n in enumerate(ns):
            x = want[n]
            if i:
                p = ns[i - 1]
                x = max(x, lo[-1] + (width[p] + width[n]) / 2 + TD_GAP_X)
            lo.append(x)
        hi = [0.0] * len(ns)
        for i in reversed(range(len(ns))):
            x = want[ns[i]]
            if i < len(ns) - 1:
                q = ns[i + 1]
                x = min(x, hi[i + 1] - (width[ns[i]] + width[q]) / 2 - TD_GAP_X)
            hi[i] = x
        for i, n in enumerate(ns):
            cx[n] = (lo[i] + hi[i]) / 2
        # Averaging can reintroduce overlap; one more forward push removes it.
        for i in range(1, len(ns)):
            a, b = ns[i - 1], ns[i]
            cx[b] = max(cx[b], cx[a] + (width[a] + width[b]) / 2 + TD_GAP_X)

    order = sorted(layers)
    for _ in range(12):
        for Ls, nbrs in ((order[1:], H.predecessors), (order[-2::-1], H.successors)):
            for L in Ls:
                ns = layers[L]
                want = {n: (sum(cx[m] for m in nbrs(n)) / len(list(nbrs(n)))
                            if list(nbrs(n)) else cx[n]) for n in ns}
                settle(ns, want)
    # A child a few points off one advisor's drop line reads as a jog in the
    # connector; line the two up, moving whichever one its row has room for.
    where = {n: (L, i) for L, ns in layers.items() for i, n in enumerate(ns)}

    def fits(n, new):
        L, i = where[n]
        ns = layers[L]
        return ((i == 0 or new - width[n] / 2
                 >= cx[ns[i - 1]] + width[ns[i - 1]] / 2 + TD_GAP_X)
                and (i == len(ns) - 1 or new + width[n] / 2
                     <= cx[ns[i + 1]] - width[ns[i + 1]] / 2 - TD_GAP_X))

    for L in order[1:]:
        for n in layers[L]:
            near = min(H.predecessors(n), key=lambda p: (abs(cx[p] - cx[n]), p), default=None)
            if near is None or not 0 < abs(cx[near] - cx[n]) <= 30:
                continue
            if fits(n, cx[near]):
                cx[n] = cx[near]
            elif fits(near, cx[n]):
                cx[near] = cx[n]
    left = min(cx[n] - width[n] / 2 for n in H)
    return {n: x - left for n, x in cx.items()}


def _td_ports(x, width, ends):
    """Port x positions along one box edge, one per tie, ordered by the far end."""
    ends = sorted(ends, key=lambda e: (e[1], e[0]))
    k = len(ends)
    step = min(TD_PORT, 0.55 * width / max(1, k - 1))
    return {name: x + (j - (k - 1) / 2) * step for j, (name, _) in enumerate(ends)}


TD_GAP_LINEAGE = 36.0   # space between lineages sharing one figure
TD_GAP_BAND = 40.0      # vertical space between bands of lineages (per_row)
TD_PANEL_PAD = 10.0     # space between a lineage and its panel frame
TD_PANEL_GUTTER = 8.0   # space between neighbouring panel frames


def _td_halves(text):
    """``text`` as two lines of near-equal length, broken at a space."""
    words = text.split(" ")
    if len(words) < 2:
        return [text]
    k = min(range(1, len(words)),
            key=lambda k: abs(len(" ".join(words[:k])) - len(" ".join(words[k:]))))
    return [" ".join(words[:k]), " ".join(words[k:])]


def lineage_topdown(comps, stem, *, caption=None, lead="", per_row=None, panels=False,
                    fs=8.0, show=True):
    """One figure holding ``comps`` side by side, each laid out on its own.

    Usually a single lineage. Several are passed when each is a plain chain:
    three one-line figures are a page of white space, and side by side they share
    the generation rows, so they read as the same kind of object. ``per_row``
    wraps them into bands of that many, stacked, each band centred on its own:
    six small lineages in one row would shrink the type below legibility.
    ``lead`` opens the notes, ahead of the generic description. ``panels`` puts
    each lineage in its own hairline frame on a regular grid, one cell per
    lineage, so a page of look-alike trees reads as separate trees.
    """
    per_row = per_row or len(comps)
    bands = [comps[i:i + per_row] for i in range(0, len(comps), per_row)]
    H = nx.DiGraph()
    H.add_nodes_from(sorted(n for c in comps for n in c))
    H.add_edges_from(sorted(e for c in comps for e in G.subgraph(c).edges()))
    sub = {n: phd_label(n) for n in H}

    def fit(fs, wrap):
        # Only a doctorate longer than its name sets the box width, so only that one breaks.
        lines = {n: (_td_halves(sub[n]) if wrap and len(sub[n]) > len(n) else [sub[n]])
                 if sub[n] else [] for n in H}
        head = measure(list(H), fs)
        small = measure({t for ls in lines.values() for t in ls}, max(fs - 1.1, MIN_PT))
        width = {n: max([head[n]] + [small[t] for t in lines[n]]) + 2 * BOX_PAD_X for n in H}
        # Layers are numbered on through the bands, so band 2's first row is
        # the row after band 1's last and the vertical pass below stays one loop.
        layer, layers, cx, bspan, first = {}, {}, {}, {}, set()
        depth = 0
        for b, band in enumerate(bands):
            span = -TD_GAP_LINEAGE
            for c in band:
                Hc = H.subgraph(sorted(c))
                lc, lsc = _td_layers(Hc)
                xc = _td_x(Hc, lsc, width)
                layer.update({n: L + depth for n, L in lc.items()})
                for L, ns in lsc.items():
                    layers.setdefault(L + depth, []).extend(ns)
                span += TD_GAP_LINEAGE
                cx.update({n: v + span for n, v in xc.items()})
                span = max(cx[n] + width[n] / 2 for n in c)
            bspan.update({n: span for c in band for n in c})
            first.add(min(layer[n] for c in band for n in c))
            depth = max(layers) + 1   # rows count from 0, so the next band starts below
        return lines, width, layer, layers, cx, bspan, first

    # The widest generation decides the type size. Shrinking stops at the print
    # minimum; a figure still too wide there breaks each doctorate onto two lines
    # instead, starting again from the full size.
    fs0 = fs
    for wrap in (False, True):
        fs = fs0
        while True:
            lines, width, layer, layers, cx, bspan, first = fit(fs, wrap)
            span = max(bspan.values())
            if span <= FIG_W_IN * 72.0 - 4 or fs <= MIN_PT + 0.4:   # legend is fs - 0.4
                break
            fs -= 0.2
        if span <= FIG_W_IN * 72.0 - 4:
            break
    assert span <= FIG_W_IN * 72.0, f"{stem} is {span:.0f} pt wide at the {MIN_PT} pt floor"

    box_h = fs * (1.9 + 0.8 * max(len(ls) for ls in lines.values()))
    assert all(layer[v] - layer[u] == 1 for u, v in H.edges()), "a tie skips a generation"
    exits = {p: _td_ports(cx[p], width[p], [(c, cx[c]) for c in H.successors(p)]) for p in H}
    entries = {c: _td_ports(cx[c], width[c], [(p, cx[p]) for p in H.predecessors(c)]) for c in H}
    top = {}
    cur = 0.0
    for L in sorted(layers):
        if L - 1 in top and L in first:   # framed panels sit one gutter apart
            cur -= 2 * TD_PANEL_PAD + TD_PANEL_GUTTER if panels else TD_GAP_BAND
        elif L - 1 in top:
            lean = max((abs(exits[u][v] - entries[v][u]) for u, v in H.edges()
                        if layer[v] == L), default=0.0)
            cur -= max(TD_GAP_Y, TD_SLOPE * lean)
        top[L] = cur
        cur -= box_h
    y = {n: top[layer[n]] - box_h / 2 for n in H}   # box centres
    xo = {n: (FIG_W_IN * 72.0 - bspan[n]) / 2 for n in H}   # centre each band
    frames = []
    if panels:   # one equal cell per lineage, the lineage centred in it
        cell_w = FIG_W_IN * 72.0 / per_row
        for band in bands:
            for k, comp in enumerate(band):
                lo = min(cx[n] - width[n] / 2 for n in comp)
                hi = max(cx[n] + width[n] / 2 for n in comp)
                shift = (k + 0.5) * cell_w - (lo + hi) / 2
                xo.update({n: shift for n in comp})
                rows = [layer[n] for n in comp]
                frames.append((k * cell_w + TD_PANEL_GUTTER / 2,
                               top[max(rows)] - box_h - TD_PANEL_PAD,
                               cell_w - TD_PANEL_GUTTER,
                               top[min(rows)] - top[max(rows)] + box_h + 2 * TD_PANEL_PAD))
    x = {n: cx[n] + xo[n] for n in H}

    legend_h = fs * 3.2 + (TD_PANEL_PAD if panels else 0.0)
    y_top, y_bottom = 3.0 + (TD_PANEL_PAD if panels else 0.0), cur - legend_h
    with econ_style():
        fig = plt.figure(figsize=figure_size("text", height=(y_top - y_bottom) / 72.0))
        ax = fig.add_axes([0, 0, 1, 1])
        ax.set_xlim(0, FIG_W_IN * 72.0)
        ax.set_ylim(y_bottom, y_top)
        ax.axis("off")

        for fx, fy, fw, fh in frames:
            ax.add_patch(plt.Rectangle((fx, fy), fw, fh, fill=False, edgecolor=RULE_COLOR,
                                       linewidth=0.6, zorder=0.5))
        for u, v in sorted(H.edges()):
            ax.annotate("", xy=(xo[v] + entries[v][u], top[layer[v]] + 0.3),
                        xytext=(xo[u] + exits[u][v], top[layer[u]] - box_h),
                        arrowprops=ARROW, zorder=3)

        for n in H:
            box_node(ax, x[n] - width[n] / 2, y[n], [n, *lines[n]],
                     width=width[n], height=box_h, inside=n in listed, fontsize=fs,
                     sub_fontsize=max(fs - 1.1, MIN_PT))
        node_legend(ax, "In the core author list", "Advisor outside the list",
                    loc="lower left", fontsize=fs - 0.4, bbox_to_anchor=(0.0, 0.0),
                    shape="box")

        listed_here = sorted(n for n in H if n in listed)
        if caption is None:
            caption = "Doctoral lineage: " + ", ".join(
                n for n in sorted(H) if H.in_degree(n) == 0)
        no_phd = sum(1 for n in H if not sub[n])
        no_year = sorted(n for n in H if sub[n] and pd.isna(phd_end.get(n)))
        notes = lead + (
            f"{len(H)} people, {len(listed_here)} of them among the 58 listed authors. Rows are "
            "generations, the first at the top. Under each name is the institution and year "
            "of that person's doctorate"
            + ("; it is not recorded for the advisor from outside the list" if no_phd == 1 else
               f"; it is not recorded for the {no_phd} advisors from outside the list"
               if no_phd else "")
            + (f"; the year is not recorded for {' and '.join(no_year)}" if no_year else "")
            + f". {TD_NOTE}"
        )
        save_figure(fig, FIGURE_DIR / stem, caption=caption, notes=notes)
    if show:
        plt.show()
        show_caption(caption, notes=notes)
    else:
        plt.close(fig)
    print(f"{stem}: {len(H)} people, {fs:.1f} pt type, {span:.0f}/{FIG_W_IN * 72:.0f} pt wide")


# Branching lineages get a figure each; the ones that are a single chain --
# one advisor, one student, one grand-student -- share the last figure.
_several = [c for c in components if len(c) >= 3 and _CORE(c) >= 2]
_is_chain = lambda c: all(G.in_degree(n) <= 1 and G.out_degree(n) <= 1 for n in c)  # noqa: E731
_trees = [c for c in _several if not _is_chain(c)]
_chains = [c for c in _several if _is_chain(c)]
for i, comp in enumerate(_trees, 1):
    lineage_topdown([comp], f"fig5alt_lineages_several_core_authors_{i}")
lineage_topdown(_chains, f"fig5alt_lineages_several_core_authors_{len(_trees) + 1}",
                caption="Doctoral lineages that run in a single line")


# Every lineage with exactly one listed author has the same shape -- that author
# and the two people who supervised them -- so they share one figure, two to a band.
_single = sorted((c for c in components if len(c) >= 3 and _CORE(c) == 1),
                 key=lambda c: sorted(c & listed))
# Listed authors whose lineage is under three people, by how many advisors they have.
_small = [G.in_degree(n) for c in components if len(c) < 3 for n in c & listed]
lineage_topdown(
    _single, "fig5alt_lineages_one_core_author", per_row=2, panels=True,
    caption="Doctoral lineages with a single author from the core list",
    lead=(
        "Lineages of at least three people containing exactly one of the 58 listed authors: "
        "every one is that author and the two people who supervised them, with no listed "
        f"author among the advisors. They reach {sum(len(c & listed) for c in _single)} of "
        f"the 58. Of the rest, {_small.count(1)} have a single recorded advisor and "
        f"{_small.count(0)} no recorded advisor at all, so their lineages fall under the "
        "three-person threshold; the remainder appear in the companion figures. "
    ),
)


### Figure 6. Ties within the core list, placed in time

Figure 5 shows structure but destroys chronology. This one restores it: horizontal position
is the advisee's year of doctorate, so every tie runs left to right and its length is the
academic generation gap.

It is restricted to ties where **both** people are in the 58-author list — the community's
own internal training relationships. That subgraph turns out to be small *and disconnected*:
nine separate chains, the largest holding four people. Each chain gets its own band, which is
the whole fix. The earlier version packed rows greedily by year across all nine chains at
once, so unrelated chains interleaved and their arrows crossed; the tangle was an artefact of
the packing, not a property of the data. Banding also gives vertical position a meaning,
which the earlier caption had to disclaim.

Drawing every person in the network with labels was tried and is illegible at any printable
size; the advisors outside the list are mostly leaves that add names without adding structure.

In [ ]:
# Figure 6. Ties within the core list, placed in time.
#
# Restricted to ties where *both* people are among the 58 authors -- the
# community's own internal training relationships. That subgraph is 22 people and
# 13 ties in 9 disconnected chains, the largest holding four. The earlier version
# packed rows greedily by year across all nine at once, so unrelated chains
# interleaved and their arrows crossed; giving each chain its own band removes
# the crossings and makes vertical position mean something.

internal = G.subgraph([n for n in G if n in listed]).copy()
internal.remove_nodes_from([n for n in internal if internal.degree(n) == 0])
chains = sorted(nx.connected_components(internal.to_undirected()),
                key=lambda c: (min(phd_year[n] for n in c), sorted(c)[0]))
assert not [n for n in internal if pd.isna(phd_year[n])], "a node here has no doctorate year"
print(f"{internal.number_of_nodes()} people, {internal.number_of_edges()} internal ties, "
      f"{len(chains)} chains of sizes {sorted((len(c) for c in chains), reverse=True)}")


def lineage_bands(stem, *, fs=8.0, show=True):
    wpt = measure(list(internal), fs)
    lo = min(phd_year[n] for n in internal)
    hi = max(phd_year[n] for n in internal)
    span = 1.0                      # smallest span whose right margin fits every label
    for n in internal:
        f = (LABEL_OFF + wpt[n] + 4.0) / AX_PT
        span = max(span, (phd_year[n] - (lo - 2) + 2.0) / (1 - f))
    unit = span / AX_PT
    xmin = lo - 2.0

    row, band_gap = fs * 1.85, fs * 0.95
    pos, rules, cursor = {}, [], 0.0
    for chain in chains:
        sub = []                    # sub-rows inside a band, only where labels collide
        for n in sorted(chain, key=lambda m: (phd_year[m], m)):
            r = 0
            while any(rr == r and phd_year[n] < phd_year[o] + (LABEL_OFF + wpt[o]) * unit + 1.0
                      for o, rr in sub):
                r += 1
            sub.append((n, r))
        for n, r in sub:
            pos[n] = (float(phd_year[n]), cursor - r * row)
        cursor -= (max(r for _, r in sub) + 1) * row + band_gap
        rules.append(cursor + band_gap / 2)
    rules.pop()

    ys = [p[1] for p in pos.values()]
    top, bottom = max(ys) + row * 0.9, min(ys) - row * 0.9
    axis_pt = 30.0
    fig_h_pt = (top - bottom) + axis_pt + 16.0

    with econ_style():
        fig = plt.figure(figsize=figure_size("text", height=fig_h_pt / 72.0))
        ax = fig.add_axes([AX_L, axis_pt / fig_h_pt, AX_W, (top - bottom) / fig_h_pt])
        ax.set_xlim(xmin, xmin + span)
        ax.set_ylim(bottom, top)
        for r in rules:
            hairline(ax, [xmin, xmin + span], [r, r], color=RULE_COLOR, linewidth=0.4, zorder=0)
        for u, v in sorted(internal.edges()):
            x0 = pos[u][0] + (LABEL_OFF + wpt[u] + PAD) * unit
            tip = pos[v][0] - (NODE_R + 0.6) * unit
            if x0 < tip - 1.0:
                ax.annotate("", xy=(tip, pos[v][1]), xytext=(x0, pos[u][1]),
                            arrowprops=ARROW, zorder=2)
            else:
                # The advisor's own label overhangs the advisee's year, so leaving from
                # the label edge would point the arrow backwards. Drop out of the node
                # instead and run in along the advisee's line.
                hairline(ax, [pos[u][0], pos[u][0]], [pos[u][1], pos[v][1]])
                ax.annotate("", xy=(tip, pos[v][1]), xytext=(pos[u][0], pos[v][1]),
                            arrowprops=ARROW, zorder=2)
        for n, (x, y) in pos.items():
            draw_node(ax, x, y, n, inside=True, fontsize=fs, unit=unit)

        ax.set_yticks([])
        ax.set_xticks(range(1970, 2021, 10))
        ax.set_xlabel("Year of doctorate")
        for side in ("left", "right", "top"):
            ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_bounds(lo - 2, hi + 2)
        caption = "Supervision ties within the core author list, by year of doctorate"
        notes = (f"{internal.number_of_nodes()} people and {internal.number_of_edges()} ties, "
                 "restricted to pairs where both advisor and advisee are among the 58 authors. "
                 f"Each band is one of the {len(chains)} separate chains this subgraph breaks "
                 "into; horizontal position is the year of the doctorate, so every tie's length "
                 "is an academic generation gap. Advisors outside the list are omitted here and "
                 "shown in the previous figure.")
        save_figure(fig, FIGURE_DIR / stem, caption=caption, notes=notes)
    if show:
        plt.show()
        show_caption(caption, notes=notes)
    else:
        plt.close(fig)
    print(f"{stem}: {len(chains)} bands over {span:.0f} years")


lineage_bands("fig6_supervision_over_time")

#### An alternative to weigh against it

One row per tie instead of one band per chain. It gives up the chaining — Stranlund appears
twice because he supervised twice — and buys total legibility: the eye never has to decide
which arrow is which, and each segment's length is read directly against the year axis.

In [ ]:
# Alternative: one row per tie rather than one band per chain.
#
# Trades the chaining for total legibility -- Stranlund appears twice because he
# supervised twice, and the eye never has to decide which arrow is which. The
# length of each segment is the gap between the two doctorates.

def tie_rows(stem, *, fs=8.0, show=True):
    ties = sorted(internal.edges(), key=lambda e: (phd_year[e[0]], phd_year[e[1]]))
    wpt = measure(list(internal), fs)
    lo = min(phd_year[n] for n in internal)
    hi = max(phd_year[n] for n in internal)
    xmin, span = lo - 4.0, (hi - lo) * 1.9    # both margins carry labels; iterate
    for _ in range(60):
        need_l = max((LABEL_OFF + wpt[u] + 3) / AX_PT * span - (phd_year[u] - xmin)
                     for u, _ in ties)
        need_r = max(phd_year[v] + (LABEL_OFF + wpt[v] + 3) / AX_PT * span - (xmin + span)
                     for _, v in ties)
        if need_l <= 0 and need_r <= 0:
            break
        xmin -= max(0.0, need_l) * 0.5
        span += max(0.0, need_l) * 0.5 + max(0.0, need_r) + 0.05
    unit = span / AX_PT

    row, axis_pt = fs * 1.95, 30.0
    fig_h_pt = len(ties) * row + row + axis_pt
    with econ_style():
        fig = plt.figure(figsize=figure_size("text", height=fig_h_pt / 72.0))
        ax = fig.add_axes([AX_L, axis_pt / fig_h_pt, AX_W, 1 - axis_pt / fig_h_pt])
        ax.set_xlim(xmin, xmin + span)
        ax.set_ylim(-len(ties) + 0.4, 0.7)
        for i, (u, v) in enumerate(ties):
            xu, xv = float(phd_year[u]), float(phd_year[v])
            ax.annotate("", xy=(xv - (NODE_R + 0.6) * unit, -i),
                        xytext=(xu + (NODE_R + 0.6) * unit, -i),
                        arrowprops=ARROW, zorder=2)
            # Both ends are listed authors here, so both nodes are filled; the
            # advisor's label hangs to the left so it stays clear of the arrow.
            for x, name, side in ((xu, u, "left"), (xv, v, "right")):
                draw_node(ax, x, -i, name, inside=True, fontsize=fs,
                          label_side=side, unit=unit)
        ax.set_yticks([])
        ax.set_xticks(range(1970, 2021, 10))
        ax.set_xlabel("Year of doctorate")
        for side in ("left", "right", "top"):
            ax.spines[side].set_visible(False)
        ax.spines["bottom"].set_bounds(lo - 2, hi + 2)
        caption = "Each supervision tie within the core author list, advisor to advisee"
        notes = (f"One row per tie: {internal.number_of_edges()} ties among "
                 f"{internal.number_of_nodes()} of the 58 authors, ordered by the advisor's "
                 "year of doctorate. Each segment spans the gap between the two doctorates. "
                 "Someone who supervised, or was supervised, more than once appears in more "
                 "than one row.")
        save_figure(fig, FIGURE_DIR / stem, caption=caption, notes=notes)
    if show:
        plt.show()
        show_caption(caption, notes=notes)
    else:
        plt.close(fig)
    print(f"{stem}: {len(ties)} ties")


tie_rows("fig6alt_tie_rows")

### Who the network puts at the centre

In [ ]:
deg = pd.DataFrame({
    "advisees": pd.Series({n: G.out_degree(n) for n in G}),
    "advisors": pd.Series({n: G.in_degree(n) for n in G}),
})
deg["total"] = deg.sum(axis=1)
deg["in_core_list"] = [n in listed for n in deg.index]
print(deg.sort_values(["advisees", "total"], ascending=False).head(10).to_string())

print("\nTies where both people are in the core author list:")
internal = supervision[
    supervision["advisor"].isin(listed) & supervision["advisee"].isin(listed)
]
print(internal[["advisor", "advisee", "relation_type", "institution", "confidence"]]
      .to_string(index=False))

## Where the training came from

Figure 5 names the advisors; this counts the institutions behind them. Grouping is on
`institution_ror`, so campuses of one university count together.

In [ ]:
# Figure 7. Where the core authors were trained.
#
# Institutions rather than advisors, because the advisor-level version has almost
# no distribution to show: 44 people outside the list supervised a doctorate in
# it, and only three of them supervised more than one. That version is drawn
# below for comparison, but it is a sentence, not a figure.
#
# The 37 institutions that trained exactly one author are left out rather than
# aggregated into a single bar: one bar of 37 against bars of 2-5 compresses
# every real bar into the left edge. The tail is stated exactly in the notes.

have_phd = authors.dropna(subset=["phd_institution"]).copy()
# Group on the ROR id rather than on the name -- campuses of one university share
# an id, so they count together -- and label each group with its canonical name.
# An institution with no ROR record falls back to grouping on its own name.
have_phd["_group"] = (have_phd["phd_institution_ror"].fillna("")
                      .mask(lambda s: s == "", have_phd["phd_institution"]))
inst_counts = (have_phd.groupby("_group")["phd_institution"]
               .agg(n="size", label="first").set_index("label")["n"].sort_index())
repeat_inst = inst_counts[inst_counts >= 2].sort_values(kind="stable")
n_single_inst = int((inst_counts == 1).sum())

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.34))
    bars = barh(ax, list(repeat_inst.index), list(repeat_inst.values), height=0.66)
    value_labels(ax, bars, fmt="{:,.0f}")
    ax.set_xlabel("Authors in the core list who took their doctorate there")
    integer_ticks(ax, "x")
    ax.set_xlim(0, repeat_inst.max() + 0.9)
    ax.tick_params(axis="y", length=0)
    caption = "Where the core authors took their doctorates"
    notes = (f"{len(have_phd)} of {len(authors)} authors have a recorded doctoral institution, "
             f"spread over {len(inst_counts)} institutions. Only the {len(repeat_inst)} that "
             f"trained two or more are shown; the other {n_single_inst} trained exactly one "
             "author each. Institutions are grouped on ROR id, so campuses of one university "
             "count together.")
    save_figure(fig, FIGURE_DIR / "fig7_phd_institutions", caption=caption, notes=notes)
plt.show()
show_caption(caption, notes=notes)

#### The advisor-level version of the same question

Kept for comparison, and to make the case for *not* using it. Forty-four people outside the
list supervised a doctorate in it and only three supervised more than one, so there is almost
no distribution to draw. This belongs in a sentence of the text, not in a figure.

In [ ]:
# The advisor-level version of the same question, for comparison. Three bars.
outside = supervision[~supervision["advisor"].isin(listed)
                      & supervision["advisee"].isin(listed)]
adv_counts = outside["advisor"].value_counts()
repeat_adv = adv_counts[adv_counts >= 2].sort_values()
n_single_adv = int((adv_counts == 1).sum())

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.24))
    bars = barh(ax, list(repeat_adv.index), list(repeat_adv.values), height=0.6)
    value_labels(ax, bars, fmt="{:,.0f}")
    ax.set_xlabel("Core-list authors supervised")
    integer_ticks(ax, "x")
    ax.set_xlim(0, repeat_adv.max() + 0.9)
    ax.tick_params(axis="y", length=0)
    caption = "Advisors outside the author list who trained more than one of its members"
    notes = (f"{len(adv_counts)} people outside the 58-author list supervised a doctorate in "
             f"it; only the {len(repeat_adv)} shown trained more than one, and the remaining "
             f"{n_single_adv} trained exactly one each. Doctoral supervision only.")
    save_figure(fig, FIGURE_DIR / "fig7alt_outside_advisors", caption=caption, notes=notes)
plt.show()
show_caption(caption, notes=notes)

## Training and placement

Figures 1 to 3 give the two margins of one question separately: where these doctorates
were taken, and where their holders now sit. Neither answers whether those are the same
people. The joint distribution is the interesting one, and it is one figure rather than
two.

### Figure 9. Country of doctorate against country of current post


In [ ]:
# Figures 9 and 10. Where people trained, and where they ended up.
#
# Figures 1-3 give the two margins of this separately: a bar chart of where the
# doctorates were taken, another of where the authors now sit. Neither says
# whether those are the same people. The joint distribution is the interesting
# one -- a literature whose authors all stayed where they trained is a different
# object from one that moved -- and it is one figure, not two.
#
# Ribbons run left to right, each as thick as the number of people it carries.
# Fill separates the two cases a reader is actually comparing: someone who is
# where they trained, and someone who is not. That is also what survives the
# grayscale proof; an alluvial whose ribbons differ only in position turns to
# texture there, and the movers are the whole point.

BLOCK_W = 1.8      # width of a stack block, in x units (the axis spans 100)
SPAN = 100.0
# A block two people tall is 13 points of paper at the scale below, and its
# label is 8.5 point type. The gap, not the block, is what stops two small
# categories printing their names on top of each other, so it is set in people
# and the figure grows to accommodate it rather than the labels being dropped.
BLOCK_GAP = 2.0    # vertical gap between blocks, in people
PEOPLE_PT = 6.5    # points of figure height per person


def _stack(names, totals):
    """Bottom edge of each block, largest at the top."""
    y, out = 0.0, {}
    for n in reversed(names):
        out[n] = y
        y += totals[n] + BLOCK_GAP
    return out, y - BLOCK_GAP


def _order(totals):
    """Biggest first, and 'Other' last however big it is -- it is a remainder,
    not a category, and ranking it among real ones invites reading it as one."""
    named = sorted((n for n in totals.index if n != "Other"), key=lambda n: (-totals[n], n))
    return named + (["Other"] if "Other" in totals.index else [])


def flow_figure(frame, left_col, right_col, stem, *, caption, notes,
                left_title, right_title, same_mask=None, fs=8.5, show=True):
    """``same_mask`` says who stayed, and it is a separate argument on purpose.

    Reading it off the plotted columns instead looks equivalent and is not: once
    a tail is pooled, "Other" on the left and "Other" on the right are different
    sets of countries, and a Norwegian doctorate now held in Switzerland would
    be drawn and counted as somebody who never moved. The mask is computed from
    the ungrouped columns and carried in.
    """
    pairs = frame[[left_col, right_col]].dropna()
    same = (pairs[left_col] == pairs[right_col]) if same_mask is None else same_mask.loc[pairs.index]
    counts = pairs.assign(_same=same).groupby([left_col, right_col, "_same"]).size()
    left_tot, right_tot = pairs[left_col].value_counts(), pairs[right_col].value_counts()
    left_order, right_order = _order(left_tot), _order(right_tot)
    left_y, left_h = _stack(left_order, left_tot)
    right_y, right_h = _stack(right_order, right_tot)
    # The two sides hold the same people but not the same number of blocks, so
    # the shorter stack is centred against the taller one. Left flush, the
    # biggest flow of all -- the one a reader looks at first -- leaves level and
    # arrives a block-gap higher, which reads as a change in quantity.
    _top = max(left_h, right_h)
    for _n in left_order:
        left_y[_n] += (_top - left_h) / 2
    for _n in right_order:
        right_y[_n] += (_top - right_h) / 2

    label = {("L", n): f"{n} ({left_tot[n]})" for n in left_order}
    label.update({("R", n): f"{n} ({right_tot[n]})" for n in right_order})
    wpt = measure(set(label.values()), fs)
    unit = AX_PT / SPAN                                  # points per x unit
    x_left = (max(wpt[label[("L", n)]] for n in left_order) + 5.0) / unit
    x_right = SPAN - (max(wpt[label[("R", n)]] for n in right_order) + 5.0) / unit - BLOCK_W

    # Ribbon ends are stacked inside each block in the order of the block they
    # point at, which is what keeps them from crossing inside their own bundle.
    geom = {}
    for n in left_order:
        cur = left_y[n] + left_tot[n]
        for key in sorted((k for k in counts.index if k[0] == n),
                          key=lambda k: (-right_y[k[1]], k[1], not k[2])):
            cur -= counts[key]
            geom[key] = {"count": int(counts[key]), "y_left": cur, "same": key[2]}
    for n in right_order:
        cur = right_y[n] + right_tot[n]
        for key in sorted((k for k in counts.index if k[1] == n),
                          key=lambda k: (-left_y[k[0]], k[0], not k[2])):
            cur -= counts[key]
            geom[key]["y_right"] = cur

    top = max(left_h, right_h)
    fig_h_pt = top * PEOPLE_PT + 44.0
    with econ_style():
        fig = plt.figure(figsize=figure_size("text", height=fig_h_pt / 72.0))
        ax = fig.add_axes([AX_L, 0.02, AX_W, 0.90])
        ax.set_xlim(0, SPAN)
        ax.set_ylim(-BLOCK_GAP, top + BLOCK_GAP)
        ax.axis("off")

        # Largest first, so a one-person ribbon is painted over a twenty-person
        # one rather than under it.
        for key, g in sorted(geom.items(), key=lambda kv: (-kv[1]["count"], kv[0])):
            flow_band(ax, x_left + BLOCK_W, x_right,
                      y_left=g["y_left"], y_right=g["y_right"], thickness=g["count"],
                      fill=GRAYS[3] if g["same"] else BAR_FILL,
                      zorder=2.0 + (1.0 / (g["count"] + 1)))

        for side, names, ys, tots, x0 in (
            ("L", left_order, left_y, left_tot, x_left),
            ("R", right_order, right_y, right_tot, x_right),
        ):
            for n in names:
                ax.add_patch(plt.Rectangle((x0, ys[n]), BLOCK_W, tots[n],
                                           facecolor=GRAYS[0], edgecolor=INK,
                                           linewidth=0.5, zorder=5))
                ax.text(x0 - 3.0 if side == "L" else x0 + BLOCK_W + 3.0,
                        ys[n] + tots[n] / 2, label[(side, n)],
                        ha="right" if side == "L" else "left", va="center",
                        fontsize=fs, color=INK, zorder=6)

        head_y = top + BLOCK_GAP * 0.8
        ax.text(x_left + BLOCK_W, head_y, left_title, ha="right", va="baseline",
                fontsize=fs - 0.4, color="#444444")
        ax.text(x_right, head_y, right_title, ha="left", va="baseline",
                fontsize=fs - 0.4, color="#444444")

        stayed = int(sum(g["count"] for g in geom.values() if g["same"]))
        handles = [
            plt.Rectangle((0, 0), 1, 1, facecolor=GRAYS[3], edgecolor=INK, linewidth=0.4,
                          label=f"Same on both sides ({stayed})"),
            plt.Rectangle((0, 0), 1, 1, facecolor=BAR_FILL, edgecolor=INK, linewidth=0.4,
                          label=f"Moved ({len(pairs) - stayed})"),
        ]
        ax.legend(handles=handles, frameon=False, loc="lower center", ncol=2,
                  fontsize=fs - 0.4, handlelength=1.4, borderaxespad=0.0,
                  bbox_to_anchor=(0.5, -0.045))
        save_figure(fig, FIGURE_DIR / stem, caption=caption, notes=notes)
    if show:
        plt.show()
        show_caption(caption, notes=notes)
    else:
        plt.close(fig)
    print(f"{stem}: {len(pairs)} people, {len(geom)} flows, {stayed} stayed, "
          f"{fig_h_pt / 72:.2f} in tall")
    return geom


# Figure 9. Country. Fourteen countries award these doctorates and eleven employ
# their holders, most of them once, so the tail is pooled. The threshold is
# applied across both sides at once: a country named on one side and folded into
# "Other" on the other would read as people vanishing.
COUNTRY_MIN = 3
_both = authors[authors["phd_country"].notna() & authors["affiliation_country"].notna()].copy()
_named = {
    c for c in set(_both["phd_country"]) | set(_both["affiliation_country"])
    if (_both["phd_country"] == c).sum() >= COUNTRY_MIN
    or (_both["affiliation_country"] == c).sum() >= COUNTRY_MIN
}
for col in ("phd_country", "affiliation_country"):
    _both[f"{col}_grp"] = _both[col].where(_both[col].isin(_named), "Other")
print(f"{len(_named)} countries named at the {COUNTRY_MIN}+ threshold: "
      f"{', '.join(sorted(_named))}")

_co = flow_figure(
    _both, "phd_country_grp", "affiliation_country_grp",
    "fig9_training_placement_country",
    same_mask=_both["phd_country"] == _both["affiliation_country"],
    left_title="Country of doctorate", right_title="Country of current post",
    caption="Where the core authors trained and where they now work",
    notes=(
        f"N = {len(_both)} authors with both a doctoral and a current country. A country is "
        f"named when it accounts for at least {COUNTRY_MIN} people on either side; the "
        f"remaining {len(_both) - _both['phd_country'].isin(_named).sum()} doctoral and "
        f"{len(_both) - _both['affiliation_country'].isin(_named).sum()} current countries are "
        f"pooled as 'Other'. Ribbon thickness is people. Blocks are ordered by size with "
        f"'Other' last; vertical position carries nothing else."
    ),
)


#### An alternative to weigh against it

The same table without the pooling, as a matrix. Written to an `alt` stem, so the
manuscript keeps pointing at Figure 9 until one of them is chosen.


In [ ]:
# Alternative to figure 9: the same question as a matrix.
#
# Worth weighing against the ribbons rather than replacing them outright, and
# the trade is a real one. The flow has to pool a tail of countries that appear
# once, because a ribbon one person thick between two blocks one person tall is
# not a mark anybody can read; the matrix does not, so all fifteen countries
# keep their names and the four that train people this list employs nowhere --
# Canada, Norway, Portugal, the Netherlands -- stay visible as empty columns.
# What it gives up is the sense of movement: twenty-six squares do not say
# "these people went there" the way a ribbon does.
#
# Rows and columns carry the same order, so staying put is exactly the diagonal.

_mat = pd.crosstab(_both["phd_country"], _both["affiliation_country"])
_countries = sorted(
    set(_both["phd_country"]) | set(_both["affiliation_country"]),
    key=lambda c: (-(int((_both["phd_country"] == c).sum())
                     + int((_both["affiliation_country"] == c).sum())), c),
)
_mat = _mat.reindex(index=_countries, columns=_countries, fill_value=0)
_row_tot = _mat.sum(axis=1)
_col_tot = _mat.sum(axis=0)
print(f"{len(_countries)} countries, {int((_mat > 0).sum().sum())} non-empty cells, "
      f"largest {int(_mat.values.max())}")


def training_matrix(stem, *, fs=7.5, show=True):
    n = len(_countries)
    cell = 0.311                                  # inches; square cells
    left, bar_w, gap, bottom = 0.98, 0.72, 0.07, 0.86
    grid = n * cell
    top_h = 0.52
    fig_w, fig_h = FIG_W_IN, bottom + grid + gap + top_h + 0.10

    with econ_style():
        fig = plt.figure(figsize=(fig_w, fig_h))
        ax = fig.add_axes([left / fig_w, bottom / fig_h, grid / fig_w, grid / fig_h])
        ax_top = fig.add_axes([left / fig_w, (bottom + grid + gap) / fig_h,
                               grid / fig_w, top_h / fig_h], sharex=ax)
        ax_right = fig.add_axes([(left + grid + gap) / fig_w, bottom / fig_h,
                                 bar_w / fig_w, grid / fig_h], sharey=ax)

        ax.set_xlim(-0.5, n - 0.5)
        ax.set_ylim(n - 0.5, -0.5)               # first country at the top
        for k in range(n):
            hairline(ax, [-0.5, n - 0.5], [k, k], color=RULE_COLOR, linewidth=0.3, zorder=0)
            hairline(ax, [k, k], [-0.5, n - 0.5], color=RULE_COLOR, linewidth=0.3, zorder=0)

        biggest = int(_mat.values.max())
        for i, row in enumerate(_countries):
            for j, col in enumerate(_countries):
                v = int(_mat.iat[i, j])
                if not v:
                    continue
                side = 0.88 * (v / biggest) ** 0.5
                stayed = row == col
                ax.add_patch(plt.Rectangle((j - side / 2, i - side / 2), side, side,
                                           facecolor=GRAYS[3] if stayed else BAR_FILL,
                                           edgecolor=INK, linewidth=0.4, zorder=3))
                # A number only where the square has room for one; elsewhere the
                # area is the value and the marginal bars carry the totals.
                if side * cell * 72 >= 11:
                    ax.text(j, i, str(v), ha="center", va="center", fontsize=fs - 1.4,
                            color=INK if stayed else "white", zorder=4)

        ax.set_xticks(range(n))
        ax.set_yticks(range(n))
        ax.set_xticklabels(_countries, rotation=90, fontsize=fs)
        ax.set_yticklabels(_countries, fontsize=fs)
        ax.tick_params(length=0, pad=2)
        for side_name in ("left", "right", "top", "bottom"):
            ax.spines[side_name].set_visible(False)

        _tb = bar(ax_top, range(n), [int(_col_tot[c]) for c in _countries], width=0.62)
        ax_top.set_ylim(0, max(_col_tot.max(), _row_tot.max()) * 1.18)
        ax_top.set_ylabel("Work\nhere", fontsize=fs - 0.5, rotation=0, ha="right", va="center")
        _rb = barh(ax_right, range(n), [int(_row_tot[c]) for c in _countries], height=0.62)
        ax_right.set_xlim(0, max(_col_tot.max(), _row_tot.max()) * 1.18)
        ax_right.set_xlabel("Trained here", fontsize=fs - 0.5)
        for a in (ax_top, ax_right):
            a.tick_params(labelbottom=False, labelleft=False, length=0)
            for side_name in ("left", "right", "top", "bottom"):
                a.spines[side_name].set_visible(False)
        value_labels(ax_top, _tb, fmt="{:.0f}", fontsize=fs - 1.4)
        value_labels(ax_right, _rb, fmt="{:.0f}", fontsize=fs - 1.4)

        caption = "Country of doctorate against country of current post"
        notes = (
            f"N = {len(_both)} authors, {int((_mat > 0).sum().sum())} non-empty cells. Square "
            f"area is people; the largest is {biggest}. Rows and columns share one order, so "
            f"the diagonal is everyone who works in the country that trained them "
            f"({int(sum(_mat.iat[k, k] for k in range(len(_countries))))} of {len(_both)}), "
            f"shown lighter. Marginal bars are the row and column totals. Unlike the ribbon "
            f"version this pools nothing, so a country that trains people the list employs "
            f"nowhere shows as an empty column."
        )
        save_figure(fig, FIGURE_DIR / stem, caption=caption, notes=notes)
    if show:
        plt.show()
        show_caption(caption, notes=notes)
    else:
        plt.close(fig)
    print(f"{stem}: {n}x{n} matrix, {fig_h:.2f} in tall")


training_matrix("fig9alt_training_placement_matrix")


### Figure 10. Field of doctorate against field of the current post


In [ ]:
# Figure 10. The same construction over research fields.
#
# No tail to pool here: six fields award these doctorates and eight employ their
# holders, and the smallest is one person. What does not fit is the names --
# "Engineering, Mathematics & Systems" spelled out on both sides leaves the
# ribbons about a third of the page -- so they are shortened for this figure
# only. Figure 3 carries the full vocabulary the crosswalk assigns.
FIELD_SHORT = {
    "Environment & Sustainability": "Environment",
    "Engineering, Mathematics & Systems": "Engineering & maths",
    "Political Science & Public Policy": "Political science",
    "Psychology & Neuroscience": "Psychology",
    "Management & Business": "Management",
    "Non-academic / applied": "Non-academic",
}

_fld = authors[authors["phd_field"].notna() & authors["affiliation_field"].notna()].copy()
for _col in ("phd_field", "affiliation_field"):
    _fld[f"{_col}_short"] = _fld[_col].map(lambda v: FIELD_SHORT.get(v, v))
_moved_fld = _fld["phd_field"] != _fld["affiliation_field"]
# Stated from the data rather than from memory: which fields appear on one side
# only. Both directions are real, and each means something different -- a field
# that trains nobody here, and a field nobody here trained in.
_only_phd = sorted(set(_fld["phd_field_short"]) - set(_fld["affiliation_field_short"]))
_only_now = sorted(set(_fld["affiliation_field_short"]) - set(_fld["phd_field_short"]))

flow_figure(
    _fld, "phd_field_short", "affiliation_field_short",
    "fig10_training_placement_field",
    same_mask=~_moved_fld,
    left_title="Field of doctorate", right_title="Field of current post",
    caption="Field of doctorate against field of the current post",
    notes=(
        f"N = {len(_fld)} authors with both a doctoral and a current field; "
        f"{int(_moved_fld.sum())} of them sit in a different field from their doctorate. "
        f"Ribbon thickness is people. Field names are shortened here -- Figure 3 carries "
        f"the full vocabulary. Blocks are ordered by size; vertical position carries "
        f"nothing else. {', '.join(_only_phd)} appears only on the doctoral side, and "
        f"{', '.join(_only_now)} only on the current-post side; those asymmetries are real "
        f"rather than missing data."
    ),
)


## Does the lineage show up in the citation record?

Every figure so far describes this group from the inside: what these 58 people study, where
they trained, who supervised whom. The rest of paper 4 describes something else — an author
co-citation network built from the bibliographies of the core corpus, where two authors are
tied when the same bibliography cites both. The two layers have never met.

They can. All 58 people in this base carry a canonical `citegraph` author id, and 51 of them
appear in that co-citation network. So a supervision tie can be asked a question it could not
be asked before: **do the field's bibliographies put an advisor and their student in the same
conversation more often than they would put two comparable strangers there?**

"Comparable" is the whole difficulty, and it is why this is a test rather than a picture.
Advisors and their students are prominent people, and prominent people are co-cited with
almost everyone — the network is dense enough that a tie beating an *average* pair would
prove nothing at all. The null used here therefore holds prominence fixed: each end of a tie
is replaced by an author cited by a similar number of bibliographies, and the tie is scored
against the pairs that substitution produces.

Two things make this a test of a real claim rather than a restatement of the data:

- **The bibliographies were written by the people being studied.** 92 of the 93 source
  bibliographies have at least one of these 58 authors on them, and one person wrote 39 of
  them. A student citing their advisor is not the field putting the two in one conversation.
  Panel B of figure 11 therefore drops, for each pair, the bibliographies either member
  wrote. Crucially it then *re-counts and re-matches inside what remains* — 485 of the 564
  authors in the network wrote none of the bibliographies, so filtering only the tie would
  compare people who lost most of their chances against people who lost none.
- **The ties are not independent.** Academic siblings share an advisor, and one advisor can
  appear in several ties, so Fisher's combination of the per-tie p-values overstates the
  evidence. The count of individually significant ties, read against a binomial, assumes only
  that each tie's own permutation p-value is valid, and is the number to quote.

The test itself lives in `citegraph.cocitation_stats` with unit tests, not in this notebook:
it is a manuscript claim, and this notebook is not under version control.


In [ ]:
# The cross-layer section. Everything above reads institutional/out/ alone; this
# reads the co-citation corpus as well, so it needs the citegraph output directory.
import itertools
import math
import unicodedata

# An editable install only reaches a *new* process: a kernel that already imported
# citegraph keeps what it loaded, so anything added to the source since then is
# missing until the kernel restarts. These imports are that check -- they fail
# loudly on a stale kernel rather than letting a figure half-draw.
from citegraph import CitationGraph
from citegraph.cocitation_stats import tie_cocitation_test
from citegraph.plotting import (
    NODE_EDGE,
    NODE_FILL,
    NODE_OPEN,
    network_axes,
    panel_title,
    place_node_labels,
)

CITEGRAPH_OUT = ROOT / "citegraph_out"

# The inclusion threshold has to match the network the rest of paper 4 reports,
# or the test and the manuscript describe different populations.
MIN_PAPERS = 3
NULL_DRAWS = 2000
TIES_PER_AUTHOR = 3
TEST_SEED = 20260915

corpus = CitationGraph.from_out_dir(CITEGRAPH_OUT)
if not corpus.has_authors:
    raise RuntimeError("Run the author-normalisation stage before this section.")

# --- names -------------------------------------------------------------------
# The institutional base is hand-curated and carries accents the extracted corpus
# lost ("Sterner, T" for Thomas Sterner, "Velez" for Vélez), so its spelling wins
# wherever it has one. Advisors from outside the base take theirs from the tie.
curated_names = dict(zip(authors["author_id"], authors["display_name"]))
for id_column, name_column in (("advisor_author_id", "advisor"), ("advisee_author_id", "advisee")):
    curated_names.update({
        row_id: row_name
        for row_id, row_name in zip(supervision[id_column], supervision[name_column])
        if isinstance(row_id, str) and row_id not in curated_names
    })


def _fold(text):
    return "".join(
        c for c in unicodedata.normalize("NFD", str(text)) if unicodedata.category(c) != "Mn"
    ).lower()


def full_name(author_id):
    return str(curated_names.get(author_id, corpus.authors["display_name"].get(author_id, author_id)))


def surname(author_id):
    """The curated spelling of the surname the corpus recorded.

    Slice the curated string rather than re-parsing it: the author stage has
    already decided where this person's surname starts, and deriving it again
    here would put a second parser in charge of compound names it settled.
    """
    extracted = str(corpus.authors["surname"].get(author_id) or full_name(author_id))
    curated = full_name(author_id)
    at = _fold(curated).rfind(_fold(extracted))
    return curated[at:at + len(extracted)] if at >= 0 else extracted


# --- the test ----------------------------------------------------------------
phd_ties = sorted({
    (advisor, advisee)
    for advisor, advisee in zip(supervision["advisor_author_id"], supervision["advisee_author_id"])
    if isinstance(advisor, str) and isinstance(advisee, str)
})
all_bibliographies = tie_cocitation_test(
    corpus, phd_ties, min_papers=MIN_PAPERS, n_draws=NULL_DRAWS, seed=TEST_SEED
)
third_party = tie_cocitation_test(
    corpus, phd_ties, min_papers=MIN_PAPERS, n_draws=NULL_DRAWS, seed=TEST_SEED,
    exclude_self_citations=True,
)

n_citing = all_bibliographies.n_citing_works
n_network = all_bibliographies.n_network_authors
cocitation_network = corpus.author_cocitation_network(min_papers=MIN_PAPERS)
base_in_network = sorted(a for a in authors["author_id"] if a in cocitation_network)

# How asymmetric is the self-citation exclusion? Every author who wrote none of
# the bibliographies loses nothing to it, which is why panel B re-matches inside
# what remains rather than filtering one side of a comparison.
wrote_a_bibliography = set(
    corpus.author_citations.loc[
        corpus.author_citations["record_id"].isin(set(corpus.edges["citing_id"])), "author_id"
    ]
)
n_never_wrote = sum(1 for a in cocitation_network if a not in wrote_a_bibliography)

# How much of the tie evidence is the pair citing each other?
volume_all = sum(t.observed for t in all_bibliographies.ties)
volume_third = sum(t.observed for t in third_party.ties)

# Ties are not independent: count how many people appear in more than one of them.
endpoints = pd.Series([x for t in all_bibliographies.ties for x in (t.source, t.target)])
repeated = endpoints.value_counts()
n_repeated = int((repeated > 1).sum())

print(f"{len(phd_ties)} doctoral ties carry an author id on both ends; "
      f"{all_bibliographies.n_ties} have both ends in the co-citation network "
      f"({n_network} authors over {n_citing} bibliographies, threshold {MIN_PAPERS}).")
for skipped in all_bibliographies.skipped:
    print(f"   not scored: {full_name(skipped.source)} → {full_name(skipped.target)} — {skipped.reason}")
print(f"\n{len(base_in_network)} of the {len(authors)} base authors are in the network; "
      f"{n_never_wrote} of the {n_network} network authors wrote none of the bibliographies.")
print(f"{n_repeated} people appear in more than one scored tie, so the ties are not independent.")

for label, result in (("all bibliographies", all_bibliographies), ("third party only", third_party)):
    print(f"\n{label}: {result.n_significant} of {result.n_ties} ties beat their own "
          f"degree-matched null at p < {result.alpha} "
          f"({result.expected_significant:.1f} expected by chance)")
print(f"\nCo-citation volume on the 16 ties: {volume_all} bibliographies in all, "
      f"{volume_third} of them written by someone other than the pair "
      f"({1 - volume_third / volume_all:.0%} is the pair citing each other).")

all_bibliographies.to_frame().sort_values("p_value")

### Figure 11. Supervision ties against comparable pairs

One row per tie. The filled marker counts the bibliographies citing both people; the open
marker is the median of the same count over pairs drawn from the two ends' prominence
strata. A tie whose filled marker sits well to the right of its open one is co-cited more
than the two people's individual visibility explains.

Panel B is the honest version of the same question, and the attenuation between the panels
is itself a finding: roughly half the co-citation volume on these ties is the pair citing
each other. What survives is the other half — other people putting the two names together.

The row order is panel A's, so a tie can be followed across. Ties are ordered by their
panel A p-value rather than by size, which is why `Bowles → Cárdenas`, the most co-cited
pair of the sixteen, sits near the bottom: at 37 and 66 citing bibliographies apiece, two
authors that prominent are co-cited 29 times by chance alone.


In [ ]:
# Figure 11. Each tie against the pairs it should be compared with.
#
# A dumbbell rather than two bar series: the quantity a reader wants is the gap
# between what this pair got and what comparable pairs get, and a gap is a thing
# a line can be. The asterisk is the econ convention for "this one clears the
# threshold on its own"; it is deliberately the only significance encoding, since
# the fills already carry observed-versus-null.
tie_rows = sorted(all_bibliographies.ties, key=lambda t: (t.p_value, -t.observed))
panel_a = {(t.source, t.target): t for t in all_bibliographies.ties}
panel_b = {(t.source, t.target): t for t in third_party.ties}
row_labels = [f"{surname(t.source)} → {surname(t.target)}" for t in tie_rows]
x_ceiling = max(t.observed for t in all_bibliographies.ties) + 3

with econ_style():
    fig, axes = plt.subplots(1, 2, figsize=figure_size("text", ratio=0.56), sharey=True)
    for ax, panel, letter, heading in (
        (axes[0], panel_a, "A", "All bibliographies"),
        (axes[1], panel_b, "B", "Third-party bibliographies only"),
    ):
        for row, tie in enumerate(tie_rows):
            y = len(tie_rows) - 1 - row
            scored = panel[(tie.source, tie.target)]
            hairline(ax, [scored.null_median, scored.observed], [y, y], zorder=2)
            ax.plot([scored.null_median], [y], marker="o", markersize=4.5, linestyle="none",
                    markerfacecolor=NODE_OPEN, markeredgecolor=NODE_EDGE, markeredgewidth=0.7,
                    zorder=3, label="Comparable pairs (median)" if row == 0 else None)
            ax.plot([scored.observed], [y], marker="o", markersize=5, linestyle="none",
                    markerfacecolor=INK, markeredgecolor=NODE_EDGE, markeredgewidth=0.7,
                    zorder=4, label="This pair" if row == 0 else None)
            if scored.p_value < all_bibliographies.alpha:
                ax.text(max(scored.observed, scored.null_median) + 0.9, y, "*",
                        ha="left", va="center", fontsize="small", color=INK)
        ax.set_yticks(range(len(tie_rows)))
        ax.set_yticklabels(list(reversed(row_labels)))
        ax.set_ylim(-0.8, len(tie_rows) - 0.2)
        ax.set_xlim(-0.8, x_ceiling)
        ax.set_xlabel("Bibliographies citing both")
        integer_ticks(ax, "x")
        panel_title(ax, letter, heading)
    axes[0].legend(frameon=False, loc="upper right", handlelength=1.0, borderaxespad=0.4,
                   fontsize="small")
    fig.tight_layout()

    fig11_caption = "Doctoral supervision ties against comparable pairs of authors"
    fig11_notes = (
        f"N = {all_bibliographies.n_ties} of the {len(supervision)} doctoral ties — those whose "
        f"advisor and advisee both enter the author co-citation network built from the "
        f"{n_citing} source bibliographies, which an author joins at {MIN_PAPERS} citing "
        f"bibliographies. Filled markers count the bibliographies citing both members of the "
        f"tie. Open markers are the median over {NULL_DRAWS:,} pairs drawn from the two ends' "
        f"prominence strata: each end replaced by an author cited by a similar number of "
        f"bibliographies (within 20%, floor one), with the tie's own members excluded. An "
        f"asterisk marks a tie at p < {all_bibliographies.alpha}, counting the observation in "
        f"its own null. Panel A uses every bibliography. Panel B drops, for each pair, the "
        f"bibliographies either member wrote, then re-counts and re-matches inside what "
        f"remains — filtering only the tie would compare authors who lost most of their "
        f"chances against the {n_never_wrote} of {n_network} network authors who wrote no "
        f"bibliography at all and so lost none. {all_bibliographies.n_significant} of "
        f"{all_bibliographies.n_ties} ties clear the threshold in panel A and "
        f"{third_party.n_significant} of {third_party.n_ties} in panel B, against "
        f"{all_bibliographies.expected_significant:.1f} expected by chance. That count is "
        f"descriptive: the ties are not independent — {n_repeated} people appear in more than "
        f"one — so neither a binomial test on it nor Fisher's combination of the per-tie "
        f"p-values is confirmatory, and the per-tie results are the evidence. "
        f"{len(all_bibliographies.skipped)} further ties are not scored because an end falls "
        f"below the inclusion threshold."
    )
    fig11_files = save_figure(
        fig, FIGURE_DIR / "fig11_supervision_cocitation",
        caption=fig11_caption, notes=fig11_notes,
    )
plt.show()
show_caption(fig11_caption, notes=fig11_notes)

### Figure 12. The same ties, on the map of who is read alongside whom

Figure 11 is the test; this is what it looks like. Positions come from co-citation — two
authors sit close when the corpus's bibliographies cite them together, relative to how often
each is cited at all — and the supervision ties are drawn on top of that arrangement. Short
arrows are the result of figure 11 made visible.

Not every arrow is short, and that is the point of drawing it rather than asserting it.
`Croson → Candelo` crosses most of the map; it is also one of the seven ties figure 11 cannot
distinguish from chance.

The layout follows the same two steps as the co-citation map in the companion notebook: ties
are normalised by how often each author is cited at all, and each author keeps only their
strongest normalised ties, because 61% of the pairs here are tied at least once and a
force-directed layout has nothing to separate in a network that dense.


In [ ]:
# Figure 12. Supervision ties laid over the co-citation arrangement.
#
# Two edge types in one drawing, so they have to be separable in grayscale: the
# co-citation backbone is a light hairline carrying the layout, and the
# supervision ties are black arrows carrying the claim. Nodes keep draw_node's
# vocabulary — filled for one of the 58, open for an advisor from outside.
#
# Node and edge iteration is sorted at construction throughout. `subgraph` and
# `set` iteration follow string hashing, and a fixed `seed=` does not save a
# layout from that: spring_layout seeds initial positions *by node index*.


def cosine_cocitation(first, second):
    """Salton's cosine between two authors' co-citation counts.

    Shared bibliographies over the geometric mean of the two authors' citing
    counts. Raw counts largely reproduce citation frequency; this asks how often
    the two are cited *together*, given how often each is cited at all.
    """
    if not cocitation_network.has_edge(first, second):
        return 0.0
    shared = cocitation_network[first][second]["weight"]
    return shared / math.sqrt(
        cocitation_network.nodes[first]["n_citing_papers"]
        * cocitation_network.nodes[second]["n_citing_papers"]
    )


scored_ties = sorted((t.source, t.target) for t in all_bibliographies.ties)
tie_people = sorted({person for tie in scored_ties for person in tie})
map_nodes = sorted(set(base_in_network) | set(tie_people))
base_ids = set(authors["author_id"])

backbone = nx.Graph()
backbone.add_nodes_from(map_nodes)
for person in map_nodes:
    ranked = sorted(
        ((cosine_cocitation(person, other), other) for other in map_nodes if other != person),
        key=lambda pair: (-pair[0], pair[1]),
    )
    for similarity, other in ranked[:TIES_PER_AUTHOR]:
        if similarity > 0:
            backbone.add_edge(person, other, similarity=similarity)
backbone_edges = sorted(backbone.edges())
n_map_pairs = len(map_nodes) * (len(map_nodes) - 1) // 2
map_density = sum(
    1 for u, v in itertools.combinations(map_nodes, 2) if cosine_cocitation(u, v) > 0
) / n_map_pairs
map_positions = nx.spring_layout(backbone, weight="similarity", seed=12, k=1.9, iterations=800)

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.72))
    for first, second in backbone_edges:
        hairline(ax, [map_positions[first][0], map_positions[second][0]],
                 [map_positions[first][1], map_positions[second][1]],
                 color=GRAYS[3], linewidth=0.5, zorder=1)
    for advisor, advisee in scored_ties:
        ax.annotate("", xy=map_positions[advisee], xytext=map_positions[advisor],
                    arrowprops=arrow_props(mutation_scale=7, color=INK, linewidth=0.9), zorder=3)
    busiest = max(cocitation_network.nodes[a]["n_citing_papers"] for a in map_nodes)
    node_sizes = {
        a: 3.0 + 5.0 * math.sqrt(cocitation_network.nodes[a]["n_citing_papers"] / busiest)
        for a in map_nodes
    }
    for person in map_nodes:
        ax.plot([map_positions[person][0]], [map_positions[person][1]], marker="o",
                linestyle="none", markersize=node_sizes[person],
                markerfacecolor=NODE_FILL if person in base_ids else NODE_OPEN,
                markeredgecolor=NODE_EDGE, markeredgewidth=0.7, zorder=4)
    network_axes(ax, equal=False)
    ax.margins(0.14)
    place_node_labels(
        ax,
        {person: surname(person) for person in tie_people},
        map_positions,
        node_sizes=node_sizes,
        order=sorted(tie_people, key=lambda a: -cocitation_network.nodes[a]["n_citing_papers"]),
        fontsize=6.5,
    )
    ax.legend(
        handles=[
            plt.Line2D([], [], marker="o", linestyle="none", markerfacecolor=NODE_FILL,
                       markeredgecolor=NODE_EDGE, markeredgewidth=0.7, markersize=5,
                       label="In the core author list"),
            plt.Line2D([], [], marker="o", linestyle="none", markerfacecolor=NODE_OPEN,
                       markeredgecolor=NODE_EDGE, markeredgewidth=0.7, markersize=5,
                       label="Advisor outside the list"),
            plt.Line2D([], [], color=INK, linewidth=0.9, linestyle="-",
                       label="Doctoral supervision"),
            plt.Line2D([], [], color=GRAYS[3], linewidth=0.5, linestyle="-",
                       label="Strongest co-citation ties"),
        ],
        frameon=False, loc="lower left", ncol=2, handlelength=1.4, borderaxespad=0.0,
        columnspacing=1.5, fontsize="small",
    )
    fig.tight_layout()

    fig12_caption = "Doctoral supervision ties on the map of who is cited alongside whom"
    fig12_notes = (
        f"{len(map_nodes)} authors: the {len(base_in_network)} of the {len(authors)} in this "
        f"base who enter the author co-citation network, plus "
        f"{len(set(tie_people) - base_ids)} advisors from outside the base who appear in a "
        f"scored tie. Positions come from a force-directed layout over co-citation among these "
        f"authors, normalised by the square root of the product of their citing-publication "
        f"counts, with each author keeping their {TIES_PER_AUTHOR} strongest normalised ties — "
        f"the light hairlines. Drawing every tie would be uninformative because "
        f"{map_density:.0%} of the pairs here are co-cited at least once. Positions carry no "
        f"units. Black arrows run advisor → advisee and are the "
        f"{len(scored_ties)} ties of figure 11; node area grows with the number of source "
        f"bibliographies citing the author. The arrangement describes co-citation *within this "
        f"group*, not position in the whole {n_network}-author network."
    )
    fig12_files = save_figure(
        fig, FIGURE_DIR / "fig12_lineage_on_cocitation",
        caption=fig12_caption, notes=fig12_notes,
    )
plt.show()
show_caption(fig12_caption, notes=fig12_notes)

### Figure 13. Prominence and academic age

A caveat figure, and the only one in this notebook that plots one quantity against another
per person. Any comparison of standing across this list has to reckon with the fact that the
people who finished their doctorates earliest have had the longest to be cited. They have.

The relationship is loose — the cloud is wide and the fit is carried partly by two people at
the ends — which is itself worth saying: seniority explains some of the spread in prominence
across this group, not most of it.


In [ ]:
# Figure 13. Doctorate year against co-citation degree.
#
# A scatter with an OLS fit, no grouping and no second encoding: the figure has
# one job, which is to show that the two quantities move together enough to be
# worth a caveat in the text. Four people are named -- the three most co-cited
# and the earliest doctorate -- because a reader will ask who the outliers are.
# Degree comes off the network this section already built, not from
# citegraph_out/author_centrality.csv. Both are `dict(network.degree())` over the
# same projection at the same threshold, but reading the CSV would make this
# notebook depend on the companion one having been run first, for a number it
# already holds.
age_vs_degree = (
    authors.loc[
        authors["phd_year_end"].notna() & authors["author_id"].isin(cocitation_network)
    ]
    .assign(degree=lambda frame: [cocitation_network.degree(a) for a in frame["author_id"]])
    .sort_values("author_id")
)
phd_year = age_vs_degree["phd_year_end"].astype(float)
degree = age_vs_degree["degree"].astype(float)
ols_slope = degree.cov(phd_year) / phd_year.var()
ols_intercept = degree.mean() - ols_slope * phd_year.mean()
age_degree_corr = float(phd_year.corr(degree))

with econ_style():
    fig, ax = plt.subplots(figsize=figure_size("text", ratio=0.6))
    span = (phd_year.min(), phd_year.max())
    ax.plot(span, [ols_intercept + ols_slope * year for year in span],
            linestyle="--", color=INK, linewidth=0.8, zorder=2)
    ax.plot(phd_year, degree, marker="o", markersize=4, linestyle="none",
            markerfacecolor=NODE_FILL, markeredgecolor=NODE_EDGE, markeredgewidth=0.7,
            zorder=3)
    named = list(dict.fromkeys(
        age_vs_degree.nlargest(3, "degree").index.tolist()
        + [age_vs_degree["phd_year_end"].idxmin()]
    ))
    place_node_labels(
        ax,
        {row: surname(age_vs_degree.loc[row, "author_id"]) for row in named},
        {row: (age_vs_degree.loc[row, "phd_year_end"], age_vs_degree.loc[row, "degree"])
         for row in age_vs_degree.index},
        node_sizes=4.0,
        order=named,
        fontsize=6.5,
    )
    ax.set_xlabel("Year of doctorate")
    ax.set_ylabel("Authors co-cited with")
    integer_ticks(ax, "y")
    ax.margins(y=0.10)
    fig.tight_layout()

    fig13_caption = "Prominence in the co-citation network against year of doctorate"
    fig13_notes = (
        f"N = {len(age_vs_degree)} of the {len(authors)} authors — those with both a recorded "
        f"year of doctorate and a place in the co-citation network. The vertical axis counts "
        f"how many of the other {n_network - 1} authors in that network each is co-cited with "
        f"at least once. The dashed line is an ordinary least-squares fit; the correlation is "
        f"{age_degree_corr:.2f}. Degree saturates at the top of a network this dense, so the "
        f"measure compresses differences among the most-cited; the companion notebook reports "
        f"the same caveat. Read the figure as a caution for any comparison of standing across "
        f"this list: part of the spread is time in the field rather than position in it."
    )
    fig13_files = save_figure(
        fig, FIGURE_DIR / "fig13_prominence_and_academic_age",
        caption=fig13_caption, notes=fig13_notes,
    )
plt.show()
show_caption(fig13_caption, notes=fig13_notes)

## Grayscale check

Every figure has to survive a black-and-white print. This writes a desaturated copy of each
PNG beside it; open them and confirm the stacked bars and the two edge styles are still
distinguishable.

In [ ]:
# The black-and-white proofs are written by save_figure itself, into figures/gray/,
# so every figure in both notebooks gets one without either having to remember.
gray = sorted((FIGURE_DIR / "gray").glob("*.gray.png"))
print(f"{len(gray)} grayscale proofs in {FIGURE_DIR / 'gray'}")
for path in gray:
    print(f"  {path.name}")


## Caveats to carry into the manuscript

1. **Supervision means doctoral supervision here.** Postdoc ties are deliberately not
   modelled, so the network understates mentorship — several of these people did postdocs
   with each other. `supervision_seed.parked.csv` holds seven researched postdoc ties if
   that decision is ever revisited.
2. **Field assignment is a judgement.** `crosswalk_fields.csv` is editable and every mapping
   is one row; revise it rather than special-casing in the notebook.
3. **One doctorate year is a correction**, not a source value: Ranehill's 2004 is recorded as
   2011 on the strength of RePEc Genealogy. It sits in the 2010s bin of Figure 4.
4. **Two second-postdoc institutions are inferred** (Anderies → CSIRO, Dreber → SIFR) from a
   department name plus a country. Both are flagged `medium` in `corrections.csv`.
5. **The co-citation test covers 16 of the 62 doctoral ties.** A tie is only testable when
   both people are cited by at least three source bibliographies, and 38 of the 56 distinct advisors
   never enter the corpus at all. The result therefore describes the ties that *could* be
   tested, and those are biased towards advisors this literature already reads. Nothing here
   says anything about the untested 46.
6. **Two generations is a collection ceiling, not a property of the field.** The base records
   each author's own supervisor, never the supervisor's supervisor, so the lineages of figures
   5, 8 and 12 cannot be deeper than they are. Do not read the depth as a finding.
7. **The supervision ties and the co-citation corpus are not independent samples.** Both were
   built around the same group: 92 of the 93 source bibliographies have one of these 58
   authors on them. Panel B of figure 11 removes each pair's own bibliographies, which is the
   strongest control the data supports, but it cannot remove a student citing a third member
   of the same lineage.
8. **Figure 13 is a caveat, not a result.** The correlation between doctorate year and
   co-citation degree is −0.37 over 48 people; seniority explains some of the spread in
   prominence across this list, not most of it, and degree saturates at the top of a network
   this dense.
